[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crunchdao/quickstarters/blob/master/competitions/causality-discovery/supervised_baseline/supervised_baseline.ipynb)

## Preliminary step

In [3]:
"""
This is a basic example of what you need to do to enter the crunch.
The code will not have access to the internet (or any socket related operation).
"""

import functools
import os
import typing
import re

import joblib
import networkx as nx
import numpy as np
import pandas as pd
import pickle
from tqdm.auto import tqdm

from scipy.stats import spearmanr, pearsonr, ttest_rel
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import roc_auc_score  # Replace with appropriate metric
from sklearn.metrics import accuracy_score, balanced_accuracy_score

from scipy.stats import energy_distance, entropy, ttest_rel, skew, kurtosis
from scipy.stats import wasserstein_distance

from scipy.spatial.distance import jensenshannon
from scipy.special import rel_entr

from statsmodels.sandbox.stats.runs import runstest_1samp
import statsmodels.api as sm	 

import pingouin as pg   # this adds .pcorr() to pandas dataframes
import dcor
import lightgbm as lgb

/home/endrem/PycharmProjects/CausalityDiscoveryCrunchDAO/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def graph_nodes_representation(graph, nodelist):
    """
    Create an alternative representation of a graph which is hashable
    and equivalent graphs have the same hash.

    Python cannot PROPERLY use nx.Graph/DiGraph as key for
    dictionaries, because two equivalent graphs with just different
    order of the nodes would result in different keys. This is
    undesirable here.

    So here we transform the graph into an equivalent form that is
    based on a specific nodelist and that is hashable. In this way,
    two equivalent graphs, once transformed, will result in identical
    keys.

    So we use the following trick: extract the adjacency matrix
    (with nodes in a fixed order) and then make a hashable thing out
    of it, through tuple(array.flatten()):
    """

    # This get the adjacency matrix with nodes in a given order, as
    # numpy array (which is not hashable):
    adjacency_matrix = nx.adjacency_matrix(graph, nodelist=nodelist).todense()

    # This transforms the numpy array into a hashable object:
    hashable = tuple(adjacency_matrix.flatten())

    return hashable

In [5]:
def create_graph_label():
    """
    Create a dictionary from graphs to labels, in two formats.
    """
    graph_label = {
        nx.DiGraph([("X", "Y"), ("v", "X"), ("v", "Y")]): "Confounder",
        nx.DiGraph([("X", "Y"), ("X", "v"), ("Y", "v")]): "Collider",
        nx.DiGraph([("X", "Y"), ("X", "v"), ("v", "Y")]): "Mediator",
        nx.DiGraph([("X", "Y"), ("v", "X")]):             "Cause of X",
        nx.DiGraph([("X", "Y"), ("v", "Y")]):             "Cause of Y",
        nx.DiGraph([("X", "Y"), ("X", "v")]):             "Consequence of X",
        nx.DiGraph([("X", "Y"), ("Y", "v")]):             "Consequence of Y",
        nx.DiGraph({"X": ["Y"], "v": []}):                "Independent",
    }

    nodelist = ["v", "X", "Y"]

    # This is an equivalent alternative to graph_label but in a form
    # for which two equivalent graphs have the same key:
    adjacency_label = {
        graph_nodes_representation(graph, nodelist): label
        for graph, label in graph_label.items()
    }

    return graph_label, adjacency_label

In [6]:
def get_labels(adjacency_matrix, adjacency_label):
    """
    Transform an adjacency_matrix (as pd.DataFrame) into a dictionary of variable:label
    """

    result = {}
    for variable in adjacency_matrix.columns.drop(["X", "Y"]):
        submatrix = adjacency_matrix.loc[[variable, "X", "Y"], [variable, "X", "Y"]]  # this is not hashable
        key = tuple(submatrix.values.flatten())  # this is hashable and a compatible with adjacency_label

        result[variable] = adjacency_label[key]

    return result

Example 1: manually creating the adjacency matrix.

In [7]:
graph_label, adjacency_label = create_graph_label()

Example 2: loading an adjacency matrix from the training set.

In [50]:
X_train = pd.read_pickle('./X_train.pkl') # dataframes already normalized to -0.999 and 0.999
y_train = pd.read_pickle('./y_train.pkl')
X_test = pd.read_pickle('./X_test.pkl')

with open('X_y_group_train.pkl', 'rb') as f:
    X_y_group_train = pickle.load(f)

with open('X_feature_mapping.pkl', 'rb') as f:
    feature_mapping = pickle.load(f)

### Correlation based features
- pearson_correlation
- spearman_correlation
- kendall_correlation
- pearson_correlation_with_rolling_multi_labeled
- partial_correlation
- partial_correlation_rank
- distance_correlation_features

In [9]:
def pearson_correlation(dataset, the_method='pearson'):
    """
    Given a dataset, we compute the correlation-based features for each
    varibale, which are the correlation between that variable with X and Y,
    as well as summary statistics (max, min, mean, std) of all pairs
    of correlations.
    """

    corr_matrix = dataset.corr(method=the_method)

    variables = dataset.columns.drop(["X", "Y"])
    df = []

    for variable in variables:
        other_corrs = corr_matrix.loc[variable].drop([variable, 'X', 'Y']).dropna() #, errors='ignore')

        corr_v_X = corr_matrix.loc[variable, 'X']
        corr_v_Y = corr_matrix.loc[variable, 'Y']

        # Handle possible NaN values (e.g., when variance is zero)
        corr_v_X = 0 if np.isnan(corr_v_X) else corr_v_X
        corr_v_Y = 0 if np.isnan(corr_v_Y) else corr_v_Y
        max_corr = 0 if other_corrs.empty else other_corrs.max()
        min_corr = 0 if other_corrs.empty else other_corrs.min()
        mean_corr = 0 if other_corrs.empty else other_corrs.mean()
        std_corr = 0 if other_corrs.empty else other_corrs.std()
        std25_corr = 0 if other_corrs.empty else np.percentile(other_corrs.values, 25)
        std75_corr = 0 if other_corrs.empty else np.percentile(other_corrs.values, 75)
        median_corr = 0 if other_corrs.empty else other_corrs.median()

        df.append({
            "variable": variable,
            f"corr(v,X)_{the_method}": corr_v_X,
            f"corr(v,Y)_{the_method}": corr_v_Y,
            f"max(corr(v, others))_{the_method}": max_corr,
            f"min(corr(v, others))_{the_method}": min_corr,
            f"mean(corr(v, others))_{the_method}": mean_corr,
            f"std(corr(v, others))_{the_method}": std_corr,
            f"std25(corr(v, others))_{the_method}": std25_corr,
            f"std75(corr(v, others))_{the_method}": std75_corr,
            f"median(corr(v, others))_{the_method}": median_corr
            #f"corr(X,Y)_{the_method}": corr_x_y
        })

    df = pd.DataFrame(df)
    df["dataset"] = dataset.name

    corr_X_Y = corr_matrix.loc['X', 'Y']
    df[f"corr(X,Y)_{the_method}"] = 0 if np.isnan(corr_X_Y) else corr_X_Y

    # Reorder columns:
    df = df[["dataset"] + [colname for colname in df.columns if colname != "dataset"]]

    return df

In [10]:
def spearman_correlation(dataset):
    return pearson_correlation(dataset, 'spearman')

In [11]:
def kendall_correlation(dataset):
    return pearson_correlation(dataset, 'kendall')

In [12]:
def pearson_correlation_with_rolling_multi_labeled(dataset, window_sizes=[300, 600, 900], meth='pearson'):
    """
    Compute rolling correlation statistics for each variable with respect to X and Y
    over rolling windows of specified sizes.
    Each statistic is labeled with _<window_size>.
    """
    variables = dataset.columns.drop(["X", "Y"])
    
    # Initialize a dictionary to hold results for each variable
    results = {variable: {"variable": variable} for variable in variables}

    for window_size in window_sizes:
        # Get the data for the current window size
        window_data = dataset.iloc[-window_size:]

        # Compute the correlation matrix once per window size
        corr_matrix = window_data.corr(method=meth)

        corr_X_Y = corr_matrix.loc['X', 'Y']
        corr_X_Y = 0 if np.isnan(corr_X_Y) else corr_X_Y

        for variable in variables:
            # Get correlations of 'variable' with others, excluding itself, 'X', and 'Y'
            if variable in corr_matrix.index:
                other_corrs = corr_matrix.loc[variable].drop([variable, 'X', 'Y'], errors='ignore')
            else:
                other_corrs = pd.Series(dtype=float)

            # Extract required correlations with 'X' and 'Y'
            corr_v_X = corr_matrix.loc[variable, 'X'] if 'X' in corr_matrix.columns else np.nan
            corr_v_Y = corr_matrix.loc[variable, 'Y'] if 'Y' in corr_matrix.columns else np.nan

            # Handle possible NaN values (e.g., when variance is zero)
            corr_v_X = 0 if np.isnan(corr_v_X) else corr_v_X
            corr_v_Y = 0 if np.isnan(corr_v_Y) else corr_v_Y

            # Drop NaN values from other_corrs
            valid_other_corrs = other_corrs.dropna()

            # Calculate statistics for other correlations
            if valid_other_corrs.empty:
                max_corr = min_corr = mean_corr = std_corr = 0
                std25_corr = std75_corr = median_corr = 0
            else:
                max_corr = valid_other_corrs.max()
                min_corr = valid_other_corrs.min()
                mean_corr = valid_other_corrs.mean()
                std_corr = valid_other_corrs.std()
                std25_corr = np.percentile(valid_other_corrs.values, 25)
                std75_corr = np.percentile(valid_other_corrs.values, 75)
                median_corr = valid_other_corrs.median()

            # Update the results dictionary for this variable
            results[variable][f"corr(v,X)_{window_size}"] = corr_v_X
            results[variable][f"corr(v,Y)_{window_size}"] = corr_v_Y
            results[variable][f"corr(X,Y)_{window_size}"] = corr_X_Y
            results[variable][f"max(corr(v, others))_{window_size}"] = max_corr
            results[variable][f"min(corr(v, others))_{window_size}"] = min_corr
            results[variable][f"mean(corr(v, others))_{window_size}"] = mean_corr
            results[variable][f"std(corr(v, others))_{window_size}"] = std_corr
            results[variable][f"25th(corr(v, others))_{window_size}"] = std25_corr
            results[variable][f"75th(corr(v, others))_{window_size}"] = std75_corr
            results[variable][f"median(corr(v, others))_{window_size}"] = median_corr

    # Convert the results dictionary to a DataFrame
    result_df = pd.DataFrame.from_dict(results, orient='index').reset_index(drop=True)
    return result_df

In [13]:
def partial_correlation(dataset, corr_threshold=0.999):
    """
    Computes the partial correlation coefficient between two variables given a third variable.
    Features:
    - partial_corr(v, X | Y)
    - partial_corr(v, Y | X)
    - partial_corr(X, Y | v)
    """
    variables = dataset.columns.drop(["X", "Y"])
    df_list = []

    for variable in variables:
        pcorr_v_X_given_Y = 0
        pcorr_v_Y_given_X = 0
        pcorr_X_Y_given_v = 0

        if (dataset[variable].std() == 0 or
            dataset["X"].std() == 0 or
            dataset["Y"].std() == 0):
            print(f"Zero variance detected in variable: {variable}")
            pcorr_v_X_given_Y = 0
            pcorr_v_Y_given_X = 0
            pcorr_X_Y_given_v = 0
        else:
            # Calculate correlation matrix to check for near-perfect correlations
            corr_matrix = dataset[[variable, 'X', 'Y']].corr().abs()
            
            # Extract the upper triangle without the diagonal
            upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
            
            # Check if any correlation exceeds the threshold
            if (upper_tri > corr_threshold).any().any():
                #print(f"Near-perfect correlation detected in variable: {variable}. Assigning partial correlations to 0.")
                pcorr_v_X_given_Y = 0
                pcorr_v_Y_given_X = 0
                pcorr_X_Y_given_v = 0
            else:
                try:
                    # Calculate partial correlation matrix
                    with np.errstate(divide='ignore', invalid='ignore'):
                        pcorr_matrix = pg.pcorr(dataset[[variable, 'X', 'Y']])
                    
                    # Extract partial correlations
                    pcorr_v_X_given_Y = pcorr_matrix.loc[variable, 'X']
                    pcorr_v_Y_given_X = pcorr_matrix.loc[variable, 'Y']
                    pcorr_X_Y_given_v = pcorr_matrix.loc['X', 'Y']
                    
                    # Handle potential infinite or NaN values
                    pcorr_v_X_given_Y = 0 if not np.isfinite(pcorr_v_X_given_Y) else pcorr_v_X_given_Y
                    pcorr_v_Y_given_X = 0 if not np.isfinite(pcorr_v_Y_given_X) else pcorr_v_Y_given_X
                    pcorr_X_Y_given_v = 0 if not np.isfinite(pcorr_X_Y_given_v) else pcorr_X_Y_given_v
                
                except Exception as e:
                    print(f"Error computing partial correlation for variable {variable}: {e}")
                    pcorr_v_X_given_Y = 0
                    pcorr_v_Y_given_X = 0
                    pcorr_X_Y_given_v = 0

        df_list.append({
            "variable": variable,
            "partial_corr(v,X|Y)": pcorr_v_X_given_Y,
            "partial_corr(v,Y|X)": pcorr_v_Y_given_X,
            "partial_corr(X,Y|v)": pcorr_X_Y_given_v
        })

    df = pd.DataFrame(df_list)
    df["dataset"] = getattr(dataset, "name", None)

    df = df[["dataset", "variable", "partial_corr(v,X|Y)", "partial_corr(v,Y|X)", "partial_corr(X,Y|v)"]]
    df.fillna(0, inplace=True)

    return df

# Example usage
if __name__ == "__main__":
    # Example dataset
    np.random.seed(42)
    data = pd.DataFrame({
        'X': np.random.randn(200),
        'Y': np.random.randn(200),
        'covar1': np.random.randn(200),
        'covar2': np.random.randn(200)
    })
    data.name = 'ExampleDataset'

    cond_dist_df = partial_correlation(data)
    print(cond_dist_df)

          dataset variable  partial_corr(v,X|Y)  partial_corr(v,Y|X)  \
0  ExampleDataset   covar1            -0.131760            -0.020130   
1  ExampleDataset   covar2             0.075778            -0.108341   

   partial_corr(X,Y|v)  
0             0.091646  
1             0.102524  


In [14]:
def partial_correlation_rank(dataset, corr_threshold=0.999):
    """
    Computes partial correlation features using Spearman's rank correlation.
    Features:
    - partial_corr_rank(v, X | Y)
    - partial_corr_rank(v, Y | X)
    - partial_corr_rank(X, Y | v)
    """
    variables = dataset.columns.drop(["X", "Y"])
    df_list = []

    for variable in variables:
        pcorr_v_X_given_Y = 0
        pcorr_v_Y_given_X = 0
        pcorr_X_Y_given_v = 0

        # Check for zero variance in variables
        if (dataset[variable].std() < 1e-8 or
            dataset["X"].std() < 1e-8 or
            dataset["Y"].std() < 1e-8):
            continue  # Correlations remain zero
        else:
                        # Calculate correlation matrix to check for near-perfect correlations
            corr_matrix = dataset[[variable, 'X', 'Y']].corr().abs()
            
            # Extract the upper triangle without the diagonal
            upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
            
            # Check if any correlation exceeds the threshold
            if (upper_tri > corr_threshold).any().any():
                #print(f"Near-perfect correlation detected in variable: {variable}. Assigning partial correlations to 0.")
                pcorr_v_X_given_Y = 0
                pcorr_v_Y_given_X = 0
                pcorr_X_Y_given_v = 0
            else:
                try:
                    pcorr_v_X_given_Y = pg.partial_corr(data=dataset, x=variable, y='X', covar='Y', method='spearman')['r'].values[0]
                    pcorr_v_Y_given_X = pg.partial_corr(data=dataset, x=variable, y='Y', covar='X', method='spearman')['r'].values[0]
                    pcorr_X_Y_given_v = pg.partial_corr(data=dataset, x='X', y='Y', covar=variable, method='spearman')['r'].values[0]
                    
                    if np.isfinite(pcorr_v_X_given_Y): pcorr_v_X_given_Y = 0 
                    if np.isfinite(pcorr_v_Y_given_X): pcorr_v_Y_given_X = 0
                    if np.isfinite(pcorr_X_Y_given_v): pcorr_X_Y_given_v = 0
                
                except Exception as e:
                    print(f"Error computing partial correlation for variable {variable}: {e}")
                    pcorr_v_X_given_Y = 0
                    pcorr_v_Y_given_X = 0
                    pcorr_X_Y_given_v = 0

        # Append results
        df_list.append({
            "variable": variable,
            "partial_corr_rank(v,X|Y)": pcorr_v_X_given_Y,
            "partial_corr_rank(v,Y|X)": pcorr_v_Y_given_X,
            "partial_corr_rank(X,Y|v)": pcorr_X_Y_given_v
        })

    df = pd.DataFrame(df_list)
    df["dataset"] = getattr(dataset, "name", None)
    df = df[["dataset", "variable", "partial_corr_rank(v,X|Y)", "partial_corr_rank(v,Y|X)", "partial_corr_rank(X,Y|v)"]]
    df.fillna(0, inplace=True)

    return df

In [15]:
def distance_correlation_features(dataset):
    """
    Computes distance-correlation-based features for each variable, including distance
    correlation with X and Y, summary statistics of distance correlation with other variables,
    and distance correlation between X and Y.
    
    Features:
    - dcor(v, X)
    - dcor(v, Y)
    - dcor(v, X)^2
    - dcor(v, Y)^2
    - max(dcor(v, others))
    - min(dcor(v, others))
    - mean(dcor(v, others))
    - std(dcor(v, others))
    - 25%(dcor(v, others))
    - 75%(dcor(v, others))
    - dcor(X, Y)
    - dcor(X, Y)^2
    """
    # Define variables excluding 'X' and 'Y'
    variables = dataset.columns.drop(["X", "Y"])
    df_list = []

    # Handle missing values by imputing with the mean
    imputer = SimpleImputer(strategy='mean')
    imputed_data = pd.DataFrame(imputer.fit_transform(dataset), columns=dataset.columns)

    # Scale the data
    scaler = StandardScaler()
    scaled_data = pd.DataFrame(scaler.fit_transform(imputed_data), columns=dataset.columns)

    # Compute distance correlation between X and Y
    dcor_X_Y = dcor.distance_correlation(scaled_data["X"], scaled_data["Y"])
    dcor_X_Y_sq = dcor_X_Y ** 2

    for variable in variables:
        # Compute distance correlation between v and X
        dcor_v_X = dcor.distance_correlation(scaled_data[variable], scaled_data["X"])
        dcor_v_X_sq = dcor_v_X ** 2

        # Compute distance correlation between v and Y
        dcor_v_Y = dcor.distance_correlation(scaled_data[variable], scaled_data["Y"])
        dcor_v_Y_sq = dcor_v_Y ** 2

        # Compute distance correlation between v and all other variables (excluding X and Y)
        other_vars = variables.drop(variable)
        dcor_v_others = []
        for other_var in other_vars:
            dcor_val = dcor.distance_correlation(scaled_data[variable], scaled_data[other_var])
            dcor_v_others.append(dcor_val)
        
        # Handle cases where there might be no other variables
        if dcor_v_others:
            max_dcor = max(dcor_v_others)
            min_dcor = min(dcor_v_others)
            mean_dcor = np.mean(dcor_v_others)
            std_dcor = np.std(dcor_v_others)
            perc25_dcor = np.percentile(dcor_v_others, 25)
            perc75_dcor = np.percentile(dcor_v_others, 75)
        else:
            max_dcor = min_dcor = mean_dcor = std_dcor = perc25_dcor = perc75_dcor = 0.0

        # Append the computed features to the list
        df_list.append({
            "variable": variable,
            "dcor(v,X)": dcor_v_X,
            "dcor(v,Y)": dcor_v_Y,
            "dcor(v,X)^2": dcor_v_X_sq,
            "dcor(v,Y)^2": dcor_v_Y_sq,
            "max(dcor(v, others))": max_dcor,
            "min(dcor(v, others))": min_dcor,
            "mean(dcor(v, others))": mean_dcor,
            "std(dcor(v, others))": std_dcor,
            "25%(dcor(v, others))": perc25_dcor,
            "75%(dcor(v, others))": perc75_dcor,
            "dcor(X,Y)": dcor_X_Y,
            "dcor(X,Y)^2": dcor_X_Y_sq
        })

    # Convert the list of dictionaries to a DataFrame
    df = pd.DataFrame(df_list)

    # Add dataset name if available
    df["dataset"] = getattr(dataset, "name", "Unknown Dataset")

    # Reorder columns to have 'dataset' first
    cols = ["dataset"] + [col for col in df.columns if col != "dataset"]
    df = df[cols]

    return df

### Information-Theoretic Features

- Mutual Information
- Conditional Mutual Information
- Copula Entropy - pycopent def copula_entropy() with copula_entropy(v, X and Y)

In [16]:
def mutual_information_and_conditional_MI(dataset):
    """
    Computes mutual-information-based features for each variable, including mutual
    information with X and Y, summary statistics of MI with other variables, and
    estimates of conditional mutual information.
    
    Conditional Mutual Information Calculations:
    - CMI(v, X | Y)
    - CMI(v, Y | X)
    - CMI(X, Y | v)
    """
    variables = dataset.columns.drop(["X", "Y"])
    df_list = []

    # Initialize the imputer to fill NaN values with the mean of each column
    imputer = SimpleImputer(strategy='mean')
    imputed_data = pd.DataFrame(imputer.fit_transform(dataset), columns=dataset.columns)


    scaler = StandardScaler()
    scaled_data = pd.DataFrame(scaler.fit_transform(imputed_data), columns=dataset.columns)

    # Compute MI of all variables with X and Y at once
    mi_with_X = mutual_info_regression(scaled_data[variables], scaled_data["X"], discrete_features=False)
    mi_with_Y = mutual_info_regression(scaled_data[variables], scaled_data["Y"], discrete_features=False)

    # Compute MI between each variable and all other variables
    mi_matrix = pd.DataFrame(index=variables, columns=variables)
    for var in variables:
        mi_matrix[var] = mutual_info_regression(scaled_data[variables], scaled_data[var], discrete_features=False)

    # Compute MI between X and Y
    mi_XY = mutual_info_regression(scaled_data[["X"]], scaled_data["Y"], discrete_features=False)[0]

    for idx, variable in enumerate(variables):
        # Get MI(v, others)
        mi_v_others = mi_matrix[variable].drop(variable)

        # Append results
        result = {
            "variable": variable,
            "MI(v,X)": mi_with_X[idx],
            "MI(v,Y)": mi_with_Y[idx],
            "max(MI(v, others))": mi_v_others.max(),
            "min(MI(v, others))": mi_v_others.min(),
            "mean(MI(v, others))": mi_v_others.mean(),
            "std(MI(v, others))": mi_v_others.std(),
        }

        # Conditional Mutual Information Calculations
        # CMI(v, X | Y) = MI(v, X, Y) - MI(v, Y)
        try:
            mi_v_X_Y = mutual_info_regression(
                scaled_data[[variable, "Y"]], scaled_data["X"], discrete_features=False
            )[0]
            cmi_v_x_given_y = mi_v_X_Y - mi_with_Y[idx]
        except:
            cmi_v_x_given_y = np.nan

        # CMI(v, Y | X) = MI(v, Y, X) - MI(v, X)
        try:
            mi_v_Y_X = mutual_info_regression(
                scaled_data[[variable, "X"]], scaled_data["Y"], discrete_features=False
            )[0]
            cmi_v_y_given_x = mi_v_Y_X - mi_with_X[idx]
        except:
            cmi_v_y_given_x = np.nan

        # CMI(X, Y | v) = MI(X, Y, v) - MI(X, v)
        try:
            mi_X_Y_v = mutual_info_regression(
                scaled_data[[ "X", variable]], scaled_data["Y"], discrete_features=False
            )[0]
            mi_X_v = mutual_info_regression(
                scaled_data[[variable]], scaled_data["X"], discrete_features=False
            )[0]
            cmi_x_y_given_v = mi_X_Y_v - mi_X_v
        except:
            cmi_x_y_given_v = np.nan

        # Add Conditional MI to the result
        result["conditional_MI(v,X|Y)"] = cmi_v_x_given_y
        result["conditional_MI(v,Y|X)"] = cmi_v_y_given_x
        result["conditional_MI(X,Y|v)"] = cmi_x_y_given_v

        df_list.append(result)

    df = pd.DataFrame(df_list)
    df["dataset"] = getattr(dataset, "name", "Unknown Dataset")
    df["MI(X,Y)"] = mi_XY

    # Reorder columns: dataset first, then MI metrics
    mi_columns = [col for col in df.columns if col not in ["dataset"]]
    df = df[["dataset"] + mi_columns]

    return df

In [17]:
#copula_entropy - copula_entropy(v,X)、copula_entropy(v,Y)

def emp_u_of_optimized(x):
    sorted_indices = np.argsort(x)
    ranks = np.empty_like(sorted_indices)
    ranks[sorted_indices] = np.arange(1, len(x) + 1)
    return ranks / len(x)

def kraskov_entropy(data, k=3, epsilon=1e-10):
    """
    Estimate the entropy using the Kraskov method.
    Parameters:
    - data: Array of shape (n_samples, n_features).
    - k: Number of nearest neighbors.
    - epsilon: Small value added to avoid log(0).
    """
    n, d = data.shape
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(data)
    distances, _ = nbrs.kneighbors(data)
    
    # Distance to the k-th nearest neighbor
    epsilons = distances[:, k]
    
    # Avoid log(0) by adding epsilon and clipping
    epsilons = np.clip(epsilons, epsilon, None)
    
    avg_log_eps = np.mean(np.log(epsilons))
    const = np.log(n - 1) + np.log(2 * np.pi) * (d / 2) - np.log(k) - 1
    return const + d * avg_log_eps

def emp_Hc_of(x, y, k=3):
    ux, uy = emp_u_of_optimized(x), emp_u_of_optimized(y)
    u = np.column_stack((ux, uy))
    return kraskov_entropy(u, k=k)

x = np.random.rand(100)
y = np.random.rand(100)

# Compute empirical copula entropy
Hc = emp_Hc_of(x, y, k=3)
print(f"Empirical Copula Entropy: {Hc}")

def compute_copula_features(dataset):
    """
    Given a dataset, this function computes the copula entropy between
    each variable v and X, Y. The resulting copula entropy values are 
    used to create features to describe/embed v, as well as the copula 
    entropy between X and Y.
    """
    variables = dataset.columns.drop(["X", "Y"])

    df = []
    for variable in variables:
        copula_vX = emp_Hc_of(dataset[variable].values, dataset["X"].values)
        copula_vY = emp_Hc_of(dataset[variable].values, dataset["Y"].values)

        df.append({
            "variable": variable,
            "copula_entropy(v,X)": copula_vX,
            "copula_entropy(v,Y)": copula_vY,
        })

    df = pd.DataFrame(df)
    df["dataset"] = dataset.name

    copula_XY = emp_Hc_of(dataset["X"].values, dataset["Y"].values)
    df["copula_entropy(X,Y)"] = copula_XY

    # Reorder columns:
    df = df[["dataset"] + [colname for colname in df.columns if colname != "dataset"]]

    return df

Empirical Copula Entropy: -0.21834588994274817


In [18]:
def kraskov_entropy(data, k=3, epsilon=1e-10):
    """
    Estimate the entropy using the Kraskov method.
    Parameters:
    - data: Array of shape (n_samples, n_features).
    - k: Number of nearest neighbors.
    - epsilon: Small value added to avoid log(0).
    """
    n, d = data.shape
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(data)
    distances, _ = nbrs.kneighbors(data)
    
    # Distance to the k-th nearest neighbor
    epsilons = distances[:, k]
    
    # Avoid log(0) by adding epsilon and clipping
    epsilons = np.clip(epsilons, epsilon, None)
    
    avg_log_eps = np.mean(np.log(epsilons))
    const = np.log(n - 1) + np.log(2 * np.pi) * (d / 2) - np.log(k) - 1
    return const + d * avg_log_eps

def joint_entropy(x, y, k=3, epsilon=1e-10):
    data = np.column_stack((x, y))
    return kraskov_entropy(data, k=k, epsilon=epsilon)

def conditional_entropy(x, y, k=3, epsilon=1e-10):
    data = np.column_stack((x, y))
    joint_ent = kraskov_entropy(data, k=k, epsilon=epsilon)
    marginal_y = kraskov_entropy(y.reshape(-1, 1), k=k, epsilon=epsilon)
    return joint_ent - marginal_y

def compute_information_metrics(dataset, k=3, epsilon=1e-10):
    variables = dataset.columns.drop(["X", "Y"])
    
    df = []
    for variable in variables:
        var = dataset[variable].values
        x = dataset["X"].values
        y = dataset["Y"].values
        
        # Joint entropies
        jh_vx = joint_entropy(var, x, k=k, epsilon=epsilon)
        jh_vy = joint_entropy(var, y, k=k, epsilon=epsilon)
        
        # Conditional entropies
        ch_vx = conditional_entropy(var, x, k=k, epsilon=epsilon)
        ch_vy = conditional_entropy(var, y, k=k, epsilon=epsilon)
        
        df.append({
            "variable": variable,
            "JH(v,X)": jh_vx,
            "JH(v,Y)": jh_vy,
            "CH(v|X)": ch_vx,
            "CH(v|Y)": ch_vy,
        })

    df = pd.DataFrame(df)
    df["dataset"] = dataset.name

    # Reorder columns
    df = df[["dataset"] + [col for col in df.columns if col != "dataset"]]
    
    return df

### Discretization-Based Features

> For Features 1-3, the original variables after discretization are input; for Features 4-7, the normalized discrete variables are input.
> 

> Reference: https://github.com/diogo149/CauseEffectPairsChallenge
> 
1. **Information Metrics**
    - JH(v,x): The joint entropy $H(v, x)$ of variable *v* and *X*.
    - JH(v,y): The joint entropy $H(v, y)$ of variable *v* and *Y*.
    - CH(v|x): The conditional entropy $H(v|x)$ of variable *v* given *X*.
    - CH(v|y): The conditional entropy $H(v|y)$ of variable *v* given *Y*.
2. **Mutual Information Metrics**
    - I(v,x): The mutual information $I(v; x)$ between variable *v* and *X*.
    - I(v,y): The mutual information $I(v; y)$ between variable *v* and *Y*.
    - AMI(v,x): The adjusted mutual information $AMI(v; x)$ between variable *v* and *X*.
    - AMI(v,y): The adjusted mutual information $AMI(v; y)$ between variable *v* and *Y*.
3. **Error Probability Metrics**
    - NPE(v,x): The normalized prediction error between variable *v* and *X*.
    - NPE(v,y): The normalized prediction error between variable *v* and *Y*.
4. **Information-Geometric Causal Inference (IGCI) Metrics**
    - IGCI(v,x): The IGCI metric based on *v* and *X*.
    - IGCI(v,y): The IGCI metric based on *v* and *Y*.
    - IGCI(x,v): The IGCI metric based on *X* and *v*.
    - IGCI(y,v): The IGCI metric based on *Y* and *v*.
5. **Moment Features**
    - MM(v² * x): The normalized joint moment $I(v^2 * x)$.
    - MM(v² * y): The normalized joint moment $I(v^2 * y)$.
    - MM(v² * x²): The normalized joint moment $I(v^2 * x^2)$.
    - MM(v² * y²): The normalized joint moment $I(v^2 * y^2)$.
    - MM(v³ * x): The normalized joint moment $I(v^3 * x)$.
    - MM(v³ * y): The normalized joint moment $I(v^3 * y)$.
6. **Fit Metrics**
    - fit(v,x): The fit metric based on *v* and *X*, calculated via polynomial fitting.
    - fit(v,y): The fit metric based on *v* and *Y*, calculated via polynomial fitting.
    - fit_error(v,x): The fit error based on *v* and *X*, measured by the standard deviation of residuals after polynomial fitting.
    - fit_error(v,y): The fit error based on *v* and *Y*, measured by the standard deviation of residuals after polynomial fitting.
    - fit_noise_entropy(v,x): The noise entropy after fitting, measuring the information content of noise for *v* and *X*.
    - fit_noise_entropy(v,y): The noise entropy after fitting, measuring the information content of noise for *v* and *Y*.
    - fit_noise_skewness(v,x): The standard deviation of noise skewness, measuring asymmetry in the noise distribution for *v* and *X*.
    - fit_noise_skewness(v,y): The standard deviation of noise skewness, measuring asymmetry in the noise distribution for *v* and *Y*.
    - fit_noise_kurtosis(v,x): The standard deviation of noise kurtosis, measuring the peakedness of the noise distribution for *v* and *X*.
    - fit_noise_kurtosis(v,y): The standard deviation of noise kurtosis, measuring the peakedness of the noise distribution for *v* and *Y*.
7. **Conditional Distribution Similarity**
    - cond_dist_sim(v,x): The conditional distribution similarity of variable *v* relative to *X*.
    - cond_dist_sim(v,y): The conditional distribution similarity of variable *v* relative to *Y*.

In [19]:
def compute_information_metrics(dataset, bins=10):
    """
    Computes information metrics for each variable in the dataset with respect to 'X' and 'Y':
    1. JH(v,X): The joint entropy H(v, X) of variable v and X.
    2. JH(v,Y): The joint entropy H(v, Y) of variable v and Y.
    3. CH(v|X): The conditional entropy H(v|X) of variable v given X.
    4. CH(v|Y): The conditional entropy H(v|Y) of variable v given Y.
    """
    variables = dataset.columns.drop(['X', 'Y'], errors='ignore')
    df_list = []

    # Handle missing values by imputing with the mean
    imputer = SimpleImputer(strategy='mean')
    imputed_data = pd.DataFrame(imputer.fit_transform(dataset), columns=dataset.columns)

    # Discretize 'X' and 'Y' into bins
    try:
        x_disc = pd.cut(imputed_data['X'], bins=bins, labels=False)
        y_disc = pd.cut(imputed_data['Y'], bins=bins, labels=False)
    except KeyError:
        x_disc = y_disc = pd.Series([np.nan]*len(imputed_data))
        print("Warning: 'X' or 'Y' not found in the dataset.")

    for variable in variables:
        try:
            v_disc = pd.cut(imputed_data[variable], bins=bins, labels=False)

            # Remove any NaN values resulting from discretization
            df = pd.DataFrame({'v': v_disc, 'X': x_disc, 'Y': y_disc}).dropna()

            # Joint entropy H(v,X)
            joint_vx_counts = df.groupby(['v', 'X']).size().values
            jh_vx = entropy(joint_vx_counts, base=2)

            # Joint entropy H(v,Y)
            joint_vy_counts = df.groupby(['v', 'Y']).size().values
            jh_vy = entropy(joint_vy_counts, base=2)

            # Entropy H(X)
            x_counts = df['X'].value_counts().values
            h_x = entropy(x_counts, base=2)

            # Entropy H(Y)
            y_counts = df['Y'].value_counts().values
            h_y = entropy(y_counts, base=2)

            # Conditional entropy H(v|X) = H(v,X) - H(X)
            ch_v_given_x = jh_vx - h_x

            # Conditional entropy H(v|Y) = H(v,Y) - H(Y)
            ch_v_given_y = jh_vy - h_y

            result = {
                'variable': variable,
                'JH(v,X)': jh_vx,
                'JH(v,Y)': jh_vy,
                'CH(v|X)': ch_v_given_x,
                'CH(v|Y)': ch_v_given_y,
            }
        except Exception as e:
            # Handle exceptions and assign 0 to results
            result = {
                'variable': variable,
                'JH(v,X)': 0,
                'JH(v,Y)': 0,
                'CH(v|X)': 0,
                'CH(v|Y)': 0,
            }
            print(f"Error processing variable '{variable}': {e}")

        df_list.append(result)

    # Convert results to DataFrame
    df_results = pd.DataFrame(df_list)

    # Add dataset name if available
    df_results['dataset'] = getattr(dataset, 'name', 'Unknown Dataset')

    # Reorder columns
    columns = ['dataset', 'variable', 'JH(v,X)', 'JH(v,Y)', 'CH(v|X)', 'CH(v|Y)']
    df_results = df_results[columns]

    return df_results

In [20]:
def normalized_prediction_error(dataset):
    """
    Computes normalized prediction errors:
    - NPE(v,X): prediction error for variable v -> X, normalized by Var(X)
    - NPE(v,Y): prediction error for variable v -> Y, normalized by Var(Y)
    """
    variables = dataset.columns.drop(["X", "Y"])
    df_list = []

    for variable in variables:
        # Initialize results
        npe_vX = 0
        npe_vY = 0

        # Check for zero variance
        if (dataset[variable].std() == 0 or
            dataset["X"].std() == 0 or
            dataset["Y"].std() == 0):
            print(f"Zero variance detected in variable: {variable}")
        else:
            # NPE(v,X)
            X = dataset["X"].dropna()
            v_data = dataset[variable].dropna()
            common_index = X.index.intersection(v_data.index)
            X = X.loc[common_index]
            v_data = v_data.loc[common_index]

            X_const = sm.add_constant(v_data)
            model_x = sm.OLS(X, X_const).fit()
            mse_x = np.mean((X - model_x.fittedvalues) ** 2)
            npe_vX = mse_x / np.var(X)

            # NPE(v,Y)
            Y = dataset["Y"].dropna()
            v_data = dataset[variable].dropna()
            common_index = Y.index.intersection(v_data.index)
            Y = Y.loc[common_index]
            v_data = v_data.loc[common_index]

            Y_const = sm.add_constant(v_data)
            model_y = sm.OLS(Y, Y_const).fit()
            mse_y = np.mean((Y - model_y.fittedvalues) ** 2)
            npe_vY = mse_y / np.var(Y)

        df_list.append({
            "variable": variable,
            "NPE(v,X)": npe_vX,
            "NPE(v,Y)": npe_vY
        })

    df = pd.DataFrame(df_list)
    df["dataset"] = getattr(dataset, "name", None)

    # Reorder columns
    df = df[["dataset", "variable", "NPE(v,X)", "NPE(v,Y)"]]

    # Fill NaN values with 0
    df.fillna(0, inplace=True)

    return df

In [21]:
def compute_igci(x_vals, y_vals):
    """
    Compute the slope-based IGCI metric using x_vals and y_vals.
    Sort by x, then compute the average log of (delta y / delta x).
    """
    # Convert to numpy arrays and drop NaNs
    x_array = np.array(x_vals).flatten()
    y_array = np.array(y_vals).flatten()
    mask = ~np.isnan(x_array) & ~np.isnan(y_array)
    x_array = x_array[mask]
    y_array = y_array[mask]

    # If fewer than 2 points, return 0
    if len(x_array) < 2:
        return 0.0

    # Sort by x
    idx = np.argsort(x_array)
    x_sorted = x_array[idx]
    y_sorted = y_array[idx]

    # Differences
    dx = np.diff(x_sorted)
    dy = np.diff(y_sorted)

    # Filter out zero diffs to avoid division by zero
    valid = (dx != 0) & (dy != 0)
    dx = dx[valid]
    dy = dy[valid]

    if len(dx) == 0:
        return 0.0

    # Compute IGCI
    sum_log = np.sum(np.log(np.abs(dy / dx)))
    igci_val = sum_log / len(dx)
    return igci_val

def igci_metrics(dataset):
    """
    Computes slope-based IGCI metrics:
    - IGCI(v,x)
    - IGCI(v,y)
    - IGCI(x,v)
    - IGCI(y,v)
    for each variable v in dataset.columns, excluding 'X' and 'Y'.
    """
    variables = dataset.columns.drop(["X", "Y"])
    df_list = []

    for variable in variables:
        # Check if there's zero variance; if so, we record IGCI metrics as 0
        if (dataset[variable].std() == 0 or
            dataset["X"].std() == 0 or
            dataset["Y"].std() == 0):
            print(f"Zero variance detected in variable: {variable}")
            igci_v_x = 0
            igci_v_y = 0
            igci_x_v = 0
            igci_y_v = 0
        else:
            igci_v_x = compute_igci(dataset[variable], dataset["X"])
            igci_v_y = compute_igci(dataset[variable], dataset["Y"])
            igci_x_v = compute_igci(dataset["X"], dataset[variable])
            igci_y_v = compute_igci(dataset["Y"], dataset[variable])

        df_list.append({
            "variable": variable,
            "IGCI(v,x)": igci_v_x,
            "IGCI(v,y)": igci_v_y,
            "IGCI(x,v)": igci_x_v,
            "IGCI(y,v)": igci_y_v
        })

    df = pd.DataFrame(df_list)
    df["dataset"] = getattr(dataset, "name", None)

    # Reorder columns
    df = df[[
        "dataset", "variable",
        "IGCI(v,x)", "IGCI(v,y)",
        "IGCI(x,v)", "IGCI(y,v)"
    ]]

    # Fill NaN values with 0
    df.fillna(0, inplace=True)

    return df

In [22]:
def compute_normalized_joint_moment(a, b):
    """
    Computes a normalized joint moment:
    I(a, b) = E[a*b] / sqrt(E[a^2] * E[b^2]),
    where E[.] is the mean.
    """
    # Convert to same index, drop NaNs
    a = a.dropna()
    b = b.dropna()
    common_index = a.index.intersection(b.index)
    a = a.loc[common_index]
    b = b.loc[common_index]

    if len(a) < 2:
        return 0.0

    mean_ab = np.mean(a * b)
    mean_a2 = np.mean(a * a)
    mean_b2 = np.mean(b * b)

    # Avoid division by zero
    if mean_a2 == 0 or mean_b2 == 0:
        return 0.0

    return mean_ab / np.sqrt(mean_a2 * mean_b2)

def moment_features(dataset):
    """
    Computes the following normalized joint moments for each variable v
    in the dataset, excluding 'X' and 'Y':
      - MM(v^2 * x)
      - MM(v^2 * y)
      - MM(v^2 * x^2)
      - MM(v^2 * y^2)
      - MM(v^3 * x)
      - MM(v^3 * y)
    """
    variables = dataset.columns.drop(["X", "Y"])
    df_list = []

    for variable in variables:
        # Check for zero variance
        if (dataset[variable].std() == 0 or
            dataset["X"].std() == 0 or
            dataset["Y"].std() == 0):
            print(f"Zero variance detected in variable: {variable}")
            vals = {
                "MM(v^2*x)": 0, "MM(v^2*y)": 0,
                "MM(v^2*x^2)": 0, "MM(v^2*y^2)": 0,
                "MM(v^3*x)": 0, "MM(v^3*y)": 0
            }
        else:
            v = dataset[variable]
            x = dataset["X"]
            y = dataset["Y"]

            # Compute v^2, v^3, x^2, y^2
            v2 = v**2
            v3 = v**3
            x2 = x**2
            y2 = y**2

            vals = {
                "MM(v^2*x)": compute_normalized_joint_moment(v2, x),
                "MM(v^2*y)": compute_normalized_joint_moment(v2, y),
                "MM(v^2*x^2)": compute_normalized_joint_moment(v2, x2),
                "MM(v^2*y^2)": compute_normalized_joint_moment(v2, y2),
                "MM(v^3*x)": compute_normalized_joint_moment(v3, x),
                "MM(v^3*y)": compute_normalized_joint_moment(v3, y)
            }

        vals["variable"] = variable
        df_list.append(vals)

    df = pd.DataFrame(df_list)
    df["dataset"] = getattr(dataset, "name", None)

    # Reorder columns
    df = df[[
        "dataset", "variable",
        "MM(v^2*x)", "MM(v^2*y)",
        "MM(v^2*x^2)", "MM(v^2*y^2)",
        "MM(v^3*x)", "MM(v^3*y)"
    ]]

    # Fill NaN values with 0
    df.fillna(0, inplace=True)

    return df

In [23]:
def _shannon_entropy(residuals, bins=100, density=True):
    """Compute the Shannon entropy of the residuals."""
    # Ensure residuals are positive
    residuals = residuals - residuals.min() + 1e-9
    probabilities, _ = np.histogram(residuals, bins=bins, density=density)
    probabilities = probabilities[probabilities > 0] # ignore empty bins
    return -np.sum(probabilities * np.log(probabilities))

def _poly_fit_metrics(x_vals, y_vals, degree=2):
    """
    Fits a polynomial of given degree: y ~ f(x).
    Returns dict with:
      - fit: 1 - SSE/SST
      - fit_error: std of residuals
      - fit_noise_entropy: Shannon entropy of residuals
      - fit_noise_skewness: skewness of residuals
      - fit_noise_kurtosis: kurtosis of residuals
    """
    # Drop missing values
    x_clean = x_vals.dropna()
    y_clean = y_vals.dropna()
    
    # Align the indices
    common_index = x_clean.index.intersection(y_clean.index)
    x_clean = x_clean.loc[common_index]
    y_clean = y_clean.loc[common_index]

    # Check for sufficient data points and variability
    if len(x_clean) < degree + 1 or x_clean.std() < 1e-6:
        return {
            "fit": 0.0,
            "fit_error": 0.0,
            "fit_noise_entropy": 0.0,
            "fit_noise_skewness": 0.0,
            "fit_noise_kurtosis": 0.0
        }

    # **Normalization Step**
    x_mean = x_clean.mean()
    x_std = x_clean.std()
    x_scaled = (x_clean - x_mean) / x_std

    # Fit polynomial on scaled x
    try:
        coeffs = np.polyfit(x_scaled, y_clean, degree, rcond=None)
    except np.RankWarning:
        print("RankWarning: Polyfit may be poorly conditioned even after scaling.")
        coeffs = np.polyfit(x_scaled, y_clean, degree, rcond=1e-4)  # Use a higher rcond to mitigate warning

    y_pred = np.polyval(coeffs, x_scaled)
    residuals = y_clean - y_pred

    # Compute fit metrics
    sse = np.sum(residuals**2)
    sst = np.sum((y_clean - np.mean(y_clean))**2)
    fit_val = 1 - sse / sst if sst != 0 else 0.0
    fit_err = np.std(residuals)
    noise_entropy = _shannon_entropy(residuals)
    noise_skew = skew(residuals) if len(residuals) > 2 else 0.0
    noise_kurt = kurtosis(residuals, fisher=False) if len(residuals) > 2 else 0.0

    return {
        "fit": fit_val,
        "fit_error": fit_err,
        "fit_noise_entropy": noise_entropy,
        "fit_noise_skewness": noise_skew,
        "fit_noise_kurtosis": noise_kurt
    }

def fit_metrics(dataset):
    """
    Computes the following features for each variable v in the dataset,
    excluding 'X' and 'Y', by polynomial fitting of X~v and Y~v:
      - fit(v,x)
      - fit(v,y)
      - fit_error(v,x)
      - fit_error(v,y)
      - fit_noise_entropy(v,x)
      - fit_noise_entropy(v,y)
      - fit_noise_skewness(v,x)
      - fit_noise_skewness(v,y)
      - fit_noise_kurtosis(v,x)
      - fit_noise_kurtosis(v,y)
    """
    variables = dataset.columns.drop(["X", "Y"])
    df_list = []

    for variable in variables:
        if (dataset[variable].std() == 0 or 
            dataset["X"].std() == 0 or 
            dataset["Y"].std() == 0):
            print(f"Zero variance detected in variable: {variable}")
            vals = {
                "fit(v,x)": 0, "fit(v,y)": 0,
                "fit_error(v,x)": 0, "fit_error(v,y)": 0,
                "fit_noise_entropy(v,x)": 0, "fit_noise_entropy(v,y)": 0,
                "fit_noise_skewness(v,x)": 0, "fit_noise_skewness(v,y)": 0,
                "fit_noise_kurtosis(v,x)": 0, "fit_noise_kurtosis(v,y)": 0
            }
        else:
            # Compute metrics for v->X
            v_x_metrics = _poly_fit_metrics(dataset[variable], dataset["X"])
            # Compute metrics for v->Y
            v_y_metrics = _poly_fit_metrics(dataset[variable], dataset["Y"])

            vals = {
                "fit(v,x)": v_x_metrics["fit"],
                "fit_error(v,x)": v_x_metrics["fit_error"],
                "fit_noise_entropy(v,x)": v_x_metrics["fit_noise_entropy"],
                "fit_noise_skewness(v,x)": v_x_metrics["fit_noise_skewness"],
                "fit_noise_kurtosis(v,x)": v_x_metrics["fit_noise_kurtosis"],

                "fit(v,y)": v_y_metrics["fit"],
                "fit_error(v,y)": v_y_metrics["fit_error"],
                "fit_noise_entropy(v,y)": v_y_metrics["fit_noise_entropy"],
                "fit_noise_skewness(v,y)": v_y_metrics["fit_noise_skewness"],
                "fit_noise_kurtosis(v,y)": v_y_metrics["fit_noise_kurtosis"]
            }

        vals["variable"] = variable
        df_list.append(vals)

    df = pd.DataFrame(df_list)
    df["dataset"] = getattr(dataset, "name", None)

    # Reorder columns
    df = df[[
        "dataset", "variable",
        "fit(v,x)", "fit_error(v,x)",
        "fit_noise_entropy(v,x)", "fit_noise_skewness(v,x)", "fit_noise_kurtosis(v,x)",
        "fit(v,y)", "fit_error(v,y)",
        "fit_noise_entropy(v,y)", "fit_noise_skewness(v,y)", "fit_noise_kurtosis(v,y)"
    ]]

    df.fillna(0, inplace=True)

    return df

In [24]:
def kl_divergence(p, q):
    """
    Compute Kullback-Leibler divergence D_KL(p || q).
    Returns 0.0 if p or q is zero everywhere or if there's mismatch in supports.
    """
    # Avoid division by zero by adding a small epsilon
    epsilon = 1e-12
    p = np.clip(p, epsilon, None)
    q = np.clip(q, epsilon, None)
    return np.sum(p * np.log(p / q))

def hellinger_distance(p, q):
    """
    Compute the Hellinger distance between two probability distributions p and q.
    """
    return np.sqrt(0.5 * np.sum((np.sqrt(p) - np.sqrt(q))**2))

def bhattacharyya_distance(p, q):
    """
    Compute the Bhattacharyya distance between distributions p and q.
    BC = sum of sqrt(p[i]*q[i]) over i
    BD = -ln(BC)
    """
    bc = np.sum(np.sqrt(p * q))
    if bc <= 1e-12:
        return 999.9  # Large distance if no overlap
    return -np.log(bc)

def emd_distance(p, q, bin_centers):
    """
    Compute the Earth Mover's distance (Wasserstein distance) between 
    two discrete distributions p and q, given bin_centers.
    """
    # Convert p, q to probabilities
    p = p / (p.sum() + 1e-12)
    q = q / (q.sum() + 1e-12)
    # Wasserstein distance expects the samples or their distribution with corresponding values
    return wasserstein_distance(bin_centers, bin_centers, p, q)

def _conditional_distribution_distance(a, b, distance_func, bins=10, b_bins=10, use_bin_centers=False):
    """
    Compute the average distance (via distance_func) between conditional distributions of b
    given bins in a, and the global distribution of b.
    If use_bin_centers=True, we also compute bin centers for EMD.
    """
    data = pd.DataFrame({'a': a, 'b': b}).dropna()
    if data.shape[0] < bins:
        return 0.0
    
    # Global distribution of b
    hist_global, bin_edges = np.histogram(data['b'], bins=b_bins, density=False)
    total_global = hist_global.sum()
    if total_global == 0:
        return 0.0
    hist_global = hist_global / total_global  # Normalize manually
    
    # Optional bin centers (for EMD)
    if use_bin_centers:
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    else:
        bin_centers = None

    # Partition 'a' into bins
    data['a_bin'] = pd.cut(data['a'], bins=bins, include_lowest=True)
    
    distances = []
    for _, group in data.groupby('a_bin', observed=False):
        hist_cond, _ = np.histogram(group['b'], bins=bin_edges, density=False)
        total_cond = hist_cond.sum()
        if total_cond == 0:
            distances.append(0.0)
            continue
        hist_cond = hist_cond / total_cond  # Normalize manually
        
        # Add a small epsilon to avoid division by zero or log issues
        epsilon = 1e-12
        hist_cond += epsilon
        hist_global_eps = hist_global + epsilon
        hist_cond /= hist_cond.sum()
        hist_global_eps /= hist_global_eps.sum()
        
        # Handle distance functions that require additional arguments
        if distance_func == emd_distance:
            if bin_centers is not None:
                # Pass bin_centers when using emd_distance
                dist_val = distance_func(hist_cond, hist_global_eps, bin_centers)
            else:
                # Handle cases where bin_centers are not available
                dist_val = distance_func(hist_cond, hist_global_eps, bin_centers=None)
        elif distance_func == wasserstein_distance:
            if bin_centers is not None:
                # Note: wasserstein_distance can accept weights
                dist_val = distance_func(bin_centers, bin_centers, u_weights=hist_cond, v_weights=hist_global_eps)
            else:
                dist_val = distance_func(hist_cond, hist_global_eps)
        else:
            dist_val = distance_func(hist_cond, hist_global_eps)
        distances.append(dist_val)
    
    return np.mean(distances) if len(distances) > 0 else 0.0

def _distance_to_similarity(avg_distance):
    """
    Convert a distance in [0, ∞) to a similarity in [0,1].
    By default, use similarity = max(0, 1 - distance).
    For some distances (e.g., KL, JS), the typical range is [0,∞),
    so a direct clamp might not always be meaningful, but is shown for simplicity.
    """
    return max(0.0, 1.0 - avg_distance)

In [25]:
def conditional_distribution_similarity_generic(
    dataset,
    distance_func=jensenshannon,
    bins=10,
    b_bins=10
):
    """
    Generic function to compute conditional distribution similarity for each variable v
    (excluding 'X' and 'Y'), using distance_func to measure difference between
    conditional distributions of b in bins of a and the overall distribution of b.
    """
    variables = dataset.columns.drop(["X", "Y"])
    df_list = []

    # If distance_func is EMD, we need bin centers
    use_bin_centers = (distance_func == emd_distance)

    for variable in variables:
        if (dataset[variable].std() == 0 or
            dataset["X"].std() == 0 or
            dataset["Y"].std() == 0):
            print(f"Zero variance detected in variable: {variable}")
            vals = {
                f"cond_dist_sim(v,x)_{distance_func.__name__}": 0.0,
                f"cond_dist_sim(v,y)_{distance_func.__name__}": 0.0
            }
        else:
            d_vx = _conditional_distribution_distance(
                dataset[variable], dataset["X"], distance_func, 
                bins=bins, b_bins=b_bins, use_bin_centers=use_bin_centers
            )
            d_vy = _conditional_distribution_distance(
                dataset[variable], dataset["Y"], distance_func,
                bins=bins, b_bins=b_bins, use_bin_centers=use_bin_centers
            )
            vals = {
                f"cond_dist_sim(v,x)_{distance_func.__name__}": _distance_to_similarity(d_vx),
                f"cond_dist_sim(v,y)_{distance_func.__name__}": _distance_to_similarity(d_vy)
            }
        vals["variable"] = variable
        df_list.append(vals)

    df = pd.DataFrame(df_list)
    df["dataset"] = getattr(dataset, "name", None)
    df = df[["dataset", "variable", f"cond_dist_sim(v,x)_{distance_func.__name__}", f"cond_dist_sim(v,y)_{distance_func.__name__}"]]
    df.fillna(0, inplace=True)
    return df

# Convenience wrappers for each distance

def conditional_distribution_similarity_jensen_shannon(dataset, bins=10, b_bins=10):
    return conditional_distribution_similarity_generic(
        dataset, distance_func=jensenshannon, bins=bins, b_bins=b_bins
    )

def conditional_distribution_similarity_kl(dataset, bins=10, b_bins=10):
    return conditional_distribution_similarity_generic(
        dataset, distance_func=kl_divergence, bins=bins, b_bins=b_bins
    )

def conditional_distribution_similarity_hellinger(dataset, bins=10, b_bins=10):
    return conditional_distribution_similarity_generic(
        dataset, distance_func=hellinger_distance, bins=bins, b_bins=b_bins
    )

def conditional_distribution_similarity_bhattacharyya(dataset, bins=10, b_bins=10):
    return conditional_distribution_similarity_generic(
        dataset, distance_func=bhattacharyya_distance, bins=bins, b_bins=b_bins
    )

def conditional_distribution_similarity_emd(dataset, bins=10, b_bins=10):
    return conditional_distribution_similarity_generic(
        dataset, distance_func=emd_distance, bins=bins, b_bins=b_bins
    )

### Feature Engineering: Regression Coefficients

1. linear_regression_feature: 
     1. $Fit ~~ X, v, v^2, v * v_{others}, v * X, cos(v), sin(v) \sim Y$
     2. $Fit ~~ v, v^2, v*v_{others}, cos(v), sin(v) \sim X$
     3. $Fit ~~ v_{others}, X, X^2, cos(X), sin(X), Y, Y^2, cos(Y), sin(Y)， X*Y \sim v$
2. Ridge Regression: introducing a penalty term used GridSearchCV to selecting alpha on np.logspace(-6, 6, 13)
3. Piecewise Linear Regression
4. Piecewise Quadratic Regression


In [ ]:
def linear_regression_feature(dataset):
    """
    For each variable v (excluding 'X' and 'Y'), fit multiple linear models:
      1) Y ~ [X, v, v^2, v*v_others, v*X, cos(v), sin(v)]
         - Separate models for each feature: X, v, v^2, v*v_other, v*X, cos(v), sin(v)
      2) X ~ [v, v^2, v*v_others, cos(v), sin(v)]
         - Separate models for each feature: v, v^2, v*v_other, cos(v), sin(v)
      3) v ~ [v_others, X, X^2, cos(X), sin(X), Y, Y^2, cos(Y), sin(Y), X*Y]
         - Separate models for each feature: each v_other, X, X^2, cos(X), sin(X), Y, Y^2, cos(Y), sin(Y), X*Y

    Returns a DataFrame with R² and MSE for each feature of each model, prefixed accordingly.
    """
    variables = dataset.columns.drop(["X", "Y"])
    
    # Convert data to float and drop NaNs
    data = dataset.copy().dropna().astype(float)
    
    results = []
    for v in variables:
        v_others = [col for col in variables if col != v]
        
        # -----------------------------
        # 1) Y ~ [X, v, v^2, v*v_others, v*X, cos(v), sin(v)]
        # -----------------------------
        y_target = data["Y"].values
        y_features = {
            "X": data["X"].values,
            "v": data[v].values,
            "v_squared": data[v].values ** 2,
            "v_X": data[v].values * data["X"].values,
            "cos_v": np.cos(data[v].values),
            "sin_v": np.sin(data[v].values)
        }
        for other in v_others:
            y_features[f"v_{other}"] = data[v].values * data[other].values
        
        for feat_name, feat_values in y_features.items():
            X = feat_values.reshape(-1, 1)
            model = LinearRegression().fit(X, y_target)
            preds = model.predict(X)
            fit_r2 = r2_score(y_target, preds)
            fit_mse = mean_squared_error(y_target, preds)
            results.append({
                "variable": v,
                "model": f"Y~{feat_name}",
                "fit_r2": fit_r2,
                "fit_mse": fit_mse
            })
        
        # -----------------------------
        # 2) X ~ [v, v^2, v*v_others, cos(v), sin(v)]
        # -----------------------------
        x_target = data["X"].values
        x_features = {
            "v": data[v].values,
            "v_squared": data[v].values ** 2,
            "cos_v": np.cos(data[v].values),
            "sin_v": np.sin(data[v].values)
        }
        for other in v_others:
            x_features[f"v_{other}"] = data[v].values * data[other].values
        
        for feat_name, feat_values in x_features.items():
            X = feat_values.reshape(-1, 1)
            model = LinearRegression().fit(X, x_target)
            preds = model.predict(X)
            fit_r2 = r2_score(x_target, preds)
            fit_mse = mean_squared_error(x_target, preds)
            results.append({
                "variable": v,
                "model": f"X~{feat_name}",
                "fit_r2": fit_r2,
                "fit_mse": fit_mse
            })
        
        # -----------------------------
        # 3) v ~ [v_others, X, X^2, cos(X), sin(X), Y, Y^2, cos(Y), sin(Y), X*Y]
        # -----------------------------
        v_target = data[v].values
        v_features = {
            "X": data["X"].values,
            "X_squared": data["X"].values ** 2,
            "cos_X": np.cos(data["X"].values),
            "sin_X": np.sin(data["X"].values),
            "Y": data["Y"].values,
            "Y_squared": data["Y"].values ** 2,
            "cos_Y": np.cos(data["Y"].values),
            "sin_Y": np.sin(data["Y"].values),
            "X_Y": data["X"].values * data["Y"].values
        }
        for other in v_others:
            v_features[other] = data[other].values
        
        for feat_name, feat_values in v_features.items():
            X = feat_values.reshape(-1, 1)
            model = LinearRegression().fit(X, v_target)
            preds = model.predict(X)
            fit_r2 = r2_score(v_target, preds)
            fit_mse = mean_squared_error(v_target, preds)
            results.append({
                "variable": v,
                "model": f"v~{feat_name}",
                "fit_r2": fit_r2,
                "fit_mse": fit_mse
            })
    
    # Create DataFrame from results
    results_df = pd.DataFrame(results)
    results_df["dataset"] = getattr(dataset, "name", "unknown_dataset")
    
    # Pivot the DataFrame to have separate columns for each model's metrics
    pivot_r2 = results_df.pivot(index=["dataset", "variable"], columns="model", values="fit_r2").add_suffix("_r2")
    pivot_mse = results_df.pivot(index=["dataset", "variable"], columns="model", values="fit_mse").add_suffix("_mse")
    
    # Combine R² and MSE into final DataFrame
    final_df = pd.concat([pivot_r2, pivot_mse], axis=1).reset_index()
    final_df.fillna(0, inplace=True)
    
    return final_df

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

def linear_regression_feature_with_ridge(dataset):
    """
    For each variable v (excluding 'X' and 'Y'), fit three Ridge regression models with
    hyperparameter tuning using GridSearchCV over alpha in np.logspace(-6, 6, 13):
    
      1) Y ~ [X, v, v^2, v * v_others, v * X, cos(v), sin(v)]
      2) X ~ [v, v^2, v * v_others, cos(v), sin(v)]
      3) v ~ [v_others, X, X^2, cos(X), sin(X), Y, Y^2, cos(Y), sin(Y), X * Y]
    
    Returns a DataFrame with R² and MSE from each fit for each variable.
    
    Parameters
    ----------
    dataset : pd.DataFrame
        Must contain columns "X" and "Y" in addition to other variables.
    
    Returns
    -------
    results_df : pd.DataFrame
        Columns: [dataset, variable,
                  fit1_r2, fit1_mse,
                  fit2_r2, fit2_mse,
                  fit3_r2, fit3_mse]
    """
    # Define the alpha grid for Ridge regression
    alpha_grid = np.logspace(-6, 6, 13)
    
    # Exclude 'X' and 'Y' from variables
    variables = dataset.columns.drop(["X", "Y"])
    
    # Convert data to float and drop NaNs
    data = dataset.copy().dropna().astype(float)
    
    results = []
    
    for v in variables:
        # Identify "other" variables (excluding X, Y, and v)
        v_others = [col for col in variables if col != v]
        
        # -----------------------------
        # 1) Y ~ [X, v, v^2, v*v_others, v*X, cos(v), sin(v)]
        # -----------------------------
        y_target = data["Y"].values  # Target
        
        # Build design matrix for Model 1
        X_design_1 = pd.DataFrame({
            "X": data["X"].values,
            "v": data[v].values,
            "v_squared": data[v].values ** 2,
            "v_X_interaction": data[v].values * data["X"].values,
            "cos_v": np.cos(data[v].values),
            "sin_v": np.sin(data[v].values)
        })
        
        # Add interaction terms v * v_others
        for other in v_others:
            X_design_1[f"v_{other}_interaction"] = data[v].values * data[other].values
        
        # Define Pipeline for Model 1: Scaling + Ridge Regression
        pipeline_1 = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge())
        ])
        
        # Setup GridSearchCV for Model 1
        grid_search_1 = GridSearchCV(
            estimator=pipeline_1,
            param_grid={"ridge__alpha": alpha_grid},
            scoring="r2",
            cv=5,
            n_jobs=-1
        )
        
        # Fit Model 1
        grid_search_1.fit(X_design_1, y_target)
        best_model_1 = grid_search_1.best_estimator_
        preds1 = best_model_1.predict(X_design_1)
        fit1_r2 = r2_score(y_target, preds1)
        fit1_mse = mean_squared_error(y_target, preds1)
        
        # -----------------------------
        # 2) X ~ [v, v^2, v*v_others, cos(v), sin(v)]
        # -----------------------------
        x_target = data["X"].values  # Target
        
        # Build design matrix for Model 2
        X_design_2 = pd.DataFrame({
            "v": data[v].values,
            "v_squared": data[v].values ** 2,
            "cos_v": np.cos(data[v].values),
            "sin_v": np.sin(data[v].values)
        })
        
        # Add interaction terms v * v_others
        for other in v_others:
            X_design_2[f"v_{other}_interaction"] = data[v].values * data[other].values
        
        # Define Pipeline for Model 2: Scaling + Ridge Regression
        pipeline_2 = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge())
        ])
        
        # Setup GridSearchCV for Model 2
        grid_search_2 = GridSearchCV(
            estimator=pipeline_2,
            param_grid={"ridge__alpha": alpha_grid},
            scoring="r2",
            cv=5,
            n_jobs=-1
        )
        
        # Fit Model 2
        grid_search_2.fit(X_design_2, x_target)
        best_model_2 = grid_search_2.best_estimator_
        preds2 = best_model_2.predict(X_design_2)
        fit2_r2 = r2_score(x_target, preds2)
        fit2_mse = mean_squared_error(x_target, preds2)
        
        # -----------------------------
        # 3) v ~ [v_others, X, X^2, cos(X), sin(X), Y, Y^2, cos(Y), sin(Y), X*Y]
        # -----------------------------
        v_target = data[v].values  # Target
        
        # Build design matrix for Model 3
        X_design_3 = pd.DataFrame({
            "X": data["X"].values,
            "X_squared": data["X"].values ** 2,
            "cos_X": np.cos(data["X"].values),
            "sin_X": np.sin(data["X"].values),
            "Y": data["Y"].values,
            "Y_squared": data["Y"].values ** 2,
            "cos_Y": np.cos(data["Y"].values),
            "sin_Y": np.sin(data["Y"].values),
            "X_Y_interaction": data["X"].values * data["Y"].values
        })
        
        # Add v_others
        for other in v_others:
            X_design_3[other] = data[other].values
        
        # Define Pipeline for Model 3: Scaling + Ridge Regression
        pipeline_3 = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge())
        ])
        
        # Setup GridSearchCV for Model 3
        grid_search_3 = GridSearchCV(
            estimator=pipeline_3,
            param_grid={"ridge__alpha": alpha_grid},
            scoring="r2",
            cv=5,
            n_jobs=-1
        )
        
        # Fit Model 3
        grid_search_3.fit(X_design_3, v_target)
        best_model_3 = grid_search_3.best_estimator_
        preds3 = best_model_3.predict(X_design_3)
        fit3_r2 = r2_score(v_target, preds3)
        fit3_mse = mean_squared_error(v_target, preds3)
        
        # Append results for this variable
        results.append({
            "variable": v,
            "ridge_fit1_r2": fit1_r2,
            "ridge_fit1_mse": fit1_mse,
            "ridge_fit2_r2": fit2_r2,
            "ridge_fit2_mse": fit2_mse,
            "ridge_fit3_r2": fit3_r2,
            "ridge_fit3_mse": fit3_mse
        })
    
    # Create results DataFrame
    results_df = pd.DataFrame(results)
    results_df["dataset"] = getattr(dataset, "name", "unknown_dataset")
    
    # Reorder columns
    col_order = [
        "dataset", "variable",
        "ridge_fit1_r2", "ridge_fit1_mse",
        "ridge_fit2_r2", "ridge_fit2_mse",
        "ridge_fit3_r2", "ridge_fit3_mse"
    ]
    results_df = results_df[col_order]
    
    return results_df

np.random.seed(42)

# Number of samples
n_samples = 100

# Generate random data for X
X = np.random.normal(loc=50, scale=10, size=n_samples)

# Generate random data for Y with some dependence on X and v1
v1 = np.random.normal(loc=5, scale=2, size=n_samples)
v2 = np.random.normal(loc=10, scale=3, size=n_samples)
v3 = np.random.normal(loc=15, scale=5, size=n_samples)

# Generate Y as a linear combination of X, v1, and some noise
Y = 2.5 * X + 1.5 * v1 - 0.8 * v2 + np.random.normal(loc=0, scale=20, size=n_samples)

# Create the DataFrame
dataset = pd.DataFrame({
    'X': X,
    'Y': Y,
    'v1': v1,
    'v2': v2,
    'v3': v3
})

# Optionally, add a 'name' attribute to the dataset for identification
dataset.name = "SampleDataset"

output_df = linear_regression_feature_with_ridge(dataset)
print(output_df)

In [29]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

def linear_regression_feature_enhanced(dataset):
    """
    Enhanced feature extraction with additional transformations and Ridge regression.
    
    For each variable v (excluding 'X' and 'Y'), fit three Ridge regression models with hyperparameter tuning:
      1) Y ~ [X, v, v^2, v*v_others, v*X, cos(v), sin(v), log(v), v^3, ...]
      2) X ~ [v, v^2, v*v_others, cos(v), sin(v), log(v), v^3, ...]
      3) v ~ [v_others, X, X^2, cos(X), sin(X), Y, Y^2, cos(Y), sin(Y), X*Y, ...]
    
    Returns a DataFrame with R² and MSE from each fit for each variable.
    """
    # Define the alpha grid for Ridge regression
    alpha_grid = np.logspace(-6, 6, 13)
    
    # Define polynomial features degree
    poly_degree = 3
    
    # Initialize PolynomialFeatures transformer
    poly = PolynomialFeatures(degree=poly_degree, include_bias=False, interaction_only=False)
    
    # Exclude 'X' and 'Y' from variables
    variables = dataset.columns.drop(["X", "Y"])
    
    # Convert data to float and drop NaNs
    data = dataset.copy().dropna().astype(float)
    
    results = []
    
    for v in variables:
        # Identify "other" variables (excluding X, Y, and v)
        v_others = [col for col in variables if col != v]
        
        # -----------------------------
        # 1) Y ~ [X, v, v^2, v*v_others, v*X, cos(v), sin(v), ...]
        # -----------------------------
        y_target = data["Y"].values  # Target
        
        # Build design matrix for Model 1
        X_design_1 = pd.DataFrame({
            "X": data["X"].values,
            "v": data[v].values,
            "v_squared": data[v].values ** 2,
            "v_X_interaction": data[v].values * data["X"].values,
            "cos_v": np.cos(data[v].values),
            "sin_v": np.sin(data[v].values),
            "log_v": np.log1p(data[v].values)  # Added log transformation
        })
        
        # Add interaction terms v * v_others
        for other in v_others:
            X_design_1[f"v_{other}_interaction"] = data[v].values * data[other].values
        
        # Add higher-degree polynomial features
        poly_features_1 = poly.fit_transform(X_design_1)
        poly_feature_names_1 = poly.get_feature_names_out(X_design_1.columns)
        X_design_1 = pd.DataFrame(poly_features_1, columns=poly_feature_names_1)
        
        # Example of adding skewness and kurtosis
        X_design_1['v_skew'] = skew(data[v].values)
        X_design_1['v_kurtosis'] = kurtosis(data[v].values)
        
        # Define Pipeline for Model 1: Scaling + Ridge Regression
        pipeline_1 = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge())
        ])
        
        # Setup GridSearchCV for Model 1
        grid_search_1 = GridSearchCV(
            estimator=pipeline_1,
            param_grid={"ridge__alpha": alpha_grid},
            scoring="r2",
            cv=5,
            n_jobs=-1
        )
        
        # Fit Model 1
        grid_search_1.fit(X_design_1, y_target)
        best_model_1 = grid_search_1.best_estimator_
        preds1 = best_model_1.predict(X_design_1)
        fit1_r2 = r2_score(y_target, preds1)
        fit1_mse = mean_squared_error(y_target, preds1)
        
        # -----------------------------
        # 2) X ~ [v, v^2, v*v_others, cos(v), sin(v), ...]
        # -----------------------------
        x_target = data["X"].values  # Target
        
        # Build design matrix for Model 2
        X_design_2 = pd.DataFrame({
            "v": data[v].values,
            "v_squared": data[v].values ** 2,
            "cos_v": np.cos(data[v].values),
            "sin_v": np.sin(data[v].values),
            "log_v": np.log1p(data[v].values)  # Added log transformation
        })
        
        # Add interaction terms v * v_others
        for other in v_others:
            X_design_2[f"v_{other}_interaction"] = data[v].values * data[other].values
        
        # Add higher-degree polynomial features
        poly_features_2 = poly.fit_transform(X_design_2)
        poly_feature_names_2 = poly.get_feature_names_out(X_design_2.columns)
        X_design_2 = pd.DataFrame(poly_features_2, columns=poly_feature_names_2)
        
        # Example of adding skewness and kurtosis
        X_design_2['v_skew'] = skew(data[v].values)
        X_design_2['v_kurtosis'] = kurtosis(data[v].values)
        
        # Define Pipeline for Model 2: Scaling + Ridge Regression
        pipeline_2 = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge())
        ])
        
        # Setup GridSearchCV for Model 2
        grid_search_2 = GridSearchCV(
            estimator=pipeline_2,
            param_grid={"ridge__alpha": alpha_grid},
            scoring="r2",
            cv=5,
            n_jobs=-1
        )
        
        # Fit Model 2
        grid_search_2.fit(X_design_2, x_target)
        best_model_2 = grid_search_2.best_estimator_
        preds2 = best_model_2.predict(X_design_2)
        fit2_r2 = r2_score(x_target, preds2)
        fit2_mse = mean_squared_error(x_target, preds2)
        
        # -----------------------------
        # 3) v ~ [v_others, X, X^2, cos(X), sin(X), Y, Y^2, cos(Y), sin(Y), X*Y, ...]
        # -----------------------------
        v_target = data[v].values  # Target
        
        # Build design matrix for Model 3
        X_design_3 = pd.DataFrame({
            "X": data["X"].values,
            "X_squared": data["X"].values ** 2,
            "cos_X": np.cos(data["X"].values),
            "sin_X": np.sin(data["X"].values),
            "Y": data["Y"].values,
            "Y_squared": data["Y"].values ** 2,
            "cos_Y": np.cos(data["Y"].values),
            "sin_Y": np.sin(data["Y"].values),
            "X_Y_interaction": data["X"].values * data["Y"].values,
            "log_X": np.log1p(data["X"].values),  # Added log transformation
            "log_Y": np.log1p(data["Y"].values)   # Added log transformation
        })
        
        # Add v_others
        for other in v_others:
            X_design_3[other] = data[other].values
        
        # Add higher-degree polynomial features
        poly_features_3 = poly.fit_transform(X_design_3)
        poly_feature_names_3 = poly.get_feature_names_out(X_design_3.columns)
        X_design_3 = pd.DataFrame(poly_features_3, columns=poly_feature_names_3)
        
        # Example of adding skewness and kurtosis
        X_design_3[f"{v}_skew"] = skew(data[v].values)
        X_design_3[f"{v}_kurtosis"] = kurtosis(data[v].values)
        
        # Define Pipeline for Model 3: Scaling + Ridge Regression
        pipeline_3 = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge())
        ])
        
        # Setup GridSearchCV for Model 3
        grid_search_3 = GridSearchCV(
            estimator=pipeline_3,
            param_grid={"ridge__alpha": alpha_grid},
            scoring="r2",
            cv=5,
            n_jobs=-1
        )
        
        # Fit Model 3
        grid_search_3.fit(X_design_3, v_target)
        best_model_3 = grid_search_3.best_estimator_
        preds3 = best_model_3.predict(X_design_3)
        fit3_r2 = r2_score(v_target, preds3)
        fit3_mse = mean_squared_error(v_target, preds3)
        
        # Append results for this variable
        results.append({
            "variable": v,
            "fit1_r2": fit1_r2,
            "fit1_mse": fit1_mse,
            "fit2_r2": fit2_r2,
            "fit2_mse": fit2_mse,
            "fit3_r2": fit3_r2,
            "fit3_mse": fit3_mse
        })
    
    # Create results DataFrame
    results_df = pd.DataFrame(results)
    results_df["dataset"] = getattr(dataset, "name", "unknown_dataset")
    
    # Reorder columns
    col_order = [
        "dataset", "variable",
        "fit1_r2", "fit1_mse",
        "fit2_r2", "fit2_mse",
        "fit3_r2", "fit3_mse"
    ]
    results_df = results_df[col_order]
    
    return results_df

### Feature Engineering - Causal Discovery Features

1. ExactSearch based on BIC score: nice approach, we gain more than 6 points from it!
2. PC Algorithm
    1. features similar to ExactSearch, such as PC(v,X), PC(X,v), PC(v,Y), PC(Y,v), PC(X,Y)
3. FCI (Fast Causal Inference) Algorithm
4. GRaSP (Greedy Relaxation and Selection of Parents)
5. DML (Double Machine Learning): about 2 points improvement (not that good as we expected) with high importance in booth xgb & lgb

### Feature Engineering - Others

1. PPS (Predictive Power Score)
2. Dimension Information
3. Grid Information
4. SEM (Structural Equation Model)
5. Cloud: Masatoshi et al (2022) Detection of Unobserved Common Cause in Discrete Data Based on the MDL Principle

In [30]:
def energy_distance_features(dataset):
    """
    Given a dataset, we compute the energy distance-based features for each
    variable, which are the energy distance between that variable with X and Y,
    as well as summary statistics (max, min, mean, std) of all pairs
    of energy distances.
    """

    variables = dataset.columns.drop(["X", "Y"])
    df = []
    
    for variable in variables:
        #energy_dists = dataset.apply(lambda col: energy_distance(dataset[variable], col))
        #energy_dists = dataset.apply(lambda col: energy_distance(dataset[variable].values[0:10], col.values[0:10]))
        energy_dists_dict = {}
        target_values = dataset[variable].values[0:-1]
        for col_name in dataset.columns:
            energy_dists_dict[col_name] = energy_distance(target_values, dataset[col_name].values[0:-1])
            
        energy_dists = pd.Series(energy_dists_dict)

        #energy_dists = energy_dists.drop([variable, "X", "Y"])

        energy_dists = energy_dists.fillna(0)

        df.append({
            "variable": variable,
            "energy_dist(v,X)": energy_distance(dataset[variable].values[0:-1], dataset["X"].values[0:-1]),
            "energy_dist(v,Y)": energy_distance(dataset[variable].values[0:-1], dataset["Y"].values[0:-1]),
            "max(energy_dist(v, others))": energy_dists.max(),
            "min(energy_dist(v, others))": energy_dists.min(),
            "mean(energy_dist(v, others))": energy_dists.mean(),
            "std(energy_dist(v, others))": energy_dists.std(),
            "std25(energy_dist(v, others))": np.percentile(energy_dists.values, 25),
            "std75(energy_dist(v, others))": np.percentile(energy_dists.values, 75),
            "median(energy_dist(v, others))": np.median(energy_dists.values)
            #"energy_dist(X,Y)": energy_dist_x_y
        })

    df = pd.DataFrame(df)
    df["dataset"] = dataset.name

    df["energy_dist(X,Y)"] = energy_distance(dataset["X"].values[0:-1], dataset["Y"].values[0:-1])

    df.fillna(0, inplace=True)

    # Reorder columns:
    df = df[["dataset"] + [colname for colname in df.columns if colname != "dataset"]]
    return df


In [31]:
if False:
    i=0
    for name, dataset in tqdm(X_train.items()):
        adataset = X_train[name]
        adataset.name = name
        if name == '00067':
            df_dataset = partial_correlation(adataset) #spearman_correlation(dataset) #pearson_correlation_with_rolling_multi_labeled(dataset)
            print("very good!")


In [32]:
def ttest(dataset, pvalue_threshold=0.05):
    """
    Given a dataset, this function computes the t-test between the
    values each variable v and X, Y. The t value and the result of the
    t-test with a given pvalue_threshold, are used to create features
    to describe/embed v, as well as the t-test result between the
    values of X and Y.
    """

    variables = dataset.columns.drop(["X", "Y"])

    df = []
    for variable in variables:
        ttest_vX = ttest_rel(dataset[variable], dataset["X"])
        ttest_vY = ttest_rel(dataset[variable], dataset["Y"])

        df.append({
            "variable": variable,
            "ttest(v,X)": ttest_vX.statistic,
            f"pvalue(ttest(v,X))<={pvalue_threshold}": (ttest_vX.pvalue <= pvalue_threshold).astype(float),
            "ttest(v,Y)": ttest_vY.statistic,
            f"pvalue(ttest(v,Y))<={pvalue_threshold}": (ttest_vY.pvalue <= pvalue_threshold).astype(float),
        })

    df = pd.DataFrame(df)
    df["dataset"] = dataset.name

    ttest_XY = ttest_rel(dataset["X"], dataset["Y"])
    df["ttest(X,Y)"] = ttest_XY.statistic
    df[f"pvalue(ttest(X,Y))<={pvalue_threshold}"] = (ttest_XY.pvalue <= pvalue_threshold).astype(float)

    # some the ttest returns NaN when the variance is 0, so we fill with 0:
    df.fillna(0, inplace=True)

    # Reorder columns:
    df = df[["dataset"] + [colname for colname in df.columns if colname != "dataset"]]

    return df

In [33]:
if False:
    data = {
        "X": np.random.rand(100),
        "Y": np.random.rand(100),
        "v1": np.random.rand(100),
        "v2": np.random.rand(100),
    }
    dataset = pd.DataFrame(data)
    dataset.name = "example_dataset"

    # Compute information metrics
    info_metrics = compute_information_metrics(dataset)
    print(info_metrics)

In [34]:
def label(adjacency_matrix):
    """
    Given a graph as adjacency_matrix, create the class labels of each variable.
    """

    adjacency_graph, adjacency_label = create_graph_label()
    labels = get_labels(adjacency_matrix, adjacency_label)
    variables = adjacency_matrix.columns.drop(["X", "Y"])

    df = pd.DataFrame({
        "variable": variables,
        "label": [labels[variable] for variable in variables],
    })
    df["dataset"] = adjacency_matrix.name

    # Reorder columns:
    df = df[["dataset"] + [colname for colname in df.columns if colname != "dataset"]]

    return df


if __name__ == "__main__":
    # Create a random adjacency matrix
    data_confounder = {
        "0": [0,  0,  0,  0,  0,  0],
        "1": [0,  0,  0,  1,  0,  0],
        "2": [1,  0,  0,  1,  0,  0],
        "Y": [1,  0,  0,  0,  0,  0],
        "4": [1,  1,  0,  1,  0,  1],
        "X": [1,  0,  0,  1,  0,  0],
    }
    adj_matrix_confounder = pd.DataFrame(data_confounder, index=["0", "1", "2","Y","4", "X"]).T
    adj_matrix_confounder.index.name = "Confounder_Dataset"
    adj_matrix_confounder.name = "Confounder_Dataset"

    # Collider: X -> Y, X -> v, Y -> v
    data_collider = {
        "v": [0, 0, 0],
        "X": [1, 0, 1],
        "Y": [0, 0, 0]
    }
    adj_matrix_collider = pd.DataFrame(data_collider, index=["v", "X", "Y"])
    adj_matrix_collider.name = "Collider_Dataset"

    # Independent: Only X -> Y
    data_independent = {
        "v": [0, 0, 1],
        "X": [0, 0, 1],
        "Y": [0, 0, 0]
    }
    adj_matrix_independent = pd.DataFrame(data_independent, index=["v", "X", "Y"])
    adj_matrix_independent.name = "Independent_Dataset"

    # List of adjacency matrices
    adjacency_matrices = [
        adj_matrix_confounder,
        #adj_matrix_collider,
        #adj_matrix_independent
    ]

    # Apply the label function to each adjacency matrix and print the results
    for adj_matrix in adjacency_matrices:
        labeled_df = label(adj_matrix)
        print(f"Labeled DataFrame for {adj_matrix.name}:")
        print(labeled_df)
        print("\n" + "-"*50 + "\n")

Labeled DataFrame for Confounder_Dataset:
              dataset variable       label
0  Confounder_Dataset        0    Collider
1  Confounder_Dataset        1  Cause of Y
2  Confounder_Dataset        2  Cause of Y
3  Confounder_Dataset        4  Confounder

--------------------------------------------------



In [35]:
def create_some_columns(names_datasets, function):
    """
    Apply an embedding function to a list of datasets.
    """

    df = []
    for name, dataset in tqdm(names_datasets.items()):
        dataset = names_datasets[name]
        dataset.name = name

        try:
            df_dataset = function(dataset)
        except ValueError as e:
            print(name, e)
            raise NotImplementedError

        df_dataset["dataset"] = name
        df.append(df_dataset)

    df = pd.concat(df, axis="index").reset_index(drop=True)
    return df

In [36]:
def create_some_columns_parallel(names_datasets, function, n_jobs=-1):
    """
    Apply an embedding function to a list of datasets.

    Parallel version.
    """

    def f(name, dataset, function):
        dataset.name = name
        df_dataset = function(dataset)
        df_dataset["dataset"] = name
        return df_dataset

    df = joblib.Parallel(n_jobs=n_jobs)(
        joblib.delayed(f)(name, dataset, function)
        for name, dataset in tqdm(names_datasets.items())
    )

    df = pd.concat(df, axis="index").reset_index(drop=True)
    return df

In [37]:
def create_all_columns(functions_names_datasets, n_jobs=-1):
    """
    given a dictionary of {function1:names, function2:names,...} apply
    the desired functions to the list of datasets and merge all of them
    in a single X_y_group dataframe.
    """

    columns = []
    for function, names_datasets in functions_names_datasets.items():
        print(f"set: {function.__name__}")

        if n_jobs != 1:
            feature_set = create_some_columns_parallel(names_datasets, function, n_jobs=n_jobs)
        else:
            feature_set = create_some_columns(names_datasets, function)

        columns.append(feature_set)

    # Merge all feature sets into a single dataframe:
    columns = functools.reduce(
        lambda left, right: pd.merge(left, right, on=["dataset", "variable"]),
        columns,
    )

    return columns

We now compute `(X_train, y_train)` from all training datasets and graphs, and store it in a single dataframe `X_y_group_train`, which also contains the name of the dataset to which each variable belongs.

In Google Colab, the cell below takes about half an hour to compute. We keep `n_jobs=1` to avoid excessive memory usage.

In [38]:
if False:
    names_datasets_train = X_train
    names_graphs_train = y_train

    print(f"Creating X_y_group_train from {len(names_datasets_train)} datasets and graphs")
    X_y_group_train = create_all_columns(
        {
            pearson_correlation: names_datasets_train,
            spearman_correlation: names_datasets_train,
            kendall_correlation: names_datasets_train,
            pearson_correlation_with_rolling_multi_labeled: names_datasets_train,
            partial_correlation: names_datasets_train,
            partial_correlation_rank: names_datasets_train,
            #mutual_information_and_conditional_MI: names_datasets_train,
            compute_copula_features: names_datasets_train,
            compute_information_metrics: names_datasets_train,
            compute_information_metrics: names_datasets_train,
            normalized_prediction_error: names_datasets_train,
            igci_metrics: names_datasets_train,
            moment_features: names_datasets_train,
            fit_metrics: names_datasets_train,
            #conditional_distribution_similarity_jensen_shannon: names_datasets_train,
            #conditional_distribution_similarity_kl: names_datasets_train,
            #conditional_distribution_similarity_hellinger: names_datasets_train,
            #conditional_distribution_similarity_bhattacharyya: names_datasets_train,
            #conditional_distribution_similarity_emd: names_datasets_train,
            #energy_distance_features: names_datasets_train,
            label: names_graphs_train
        },
        n_jobs=-1,
    )

In [39]:
if False:
    print("Adding numeric labels y")
    le = LabelEncoder()
    X_y_group_train["y"] = le.fit_transform(X_y_group_train["label"])
    # reordering columns:
    X_y_group_train = X_y_group_train[["dataset", "variable"] + X_y_group_train.columns.drop(["dataset", "variable", "label", "y"]).tolist() + ["label", "y"]]
    #display(X_y_group_train)

    print("Extracting X_train, y_train, and group")
    X_train_feat = X_y_group_train.drop(["variable", "dataset", "label", "y"], axis="columns")
    y_train_feat = X_y_group_train["y"]
    group_train = X_y_group_train["dataset"]
    print(X_train_feat.shape)

In [40]:
#GAE
# A gradient-based algorithm using graph autoencoder to model non-linear causal relationships

#weighted_random_dag = DAG.erdos_renyi(n_nodes=10, n_edges=20, weight_range=(0.5, 2.0), seed=1)
#dataset = IIDSimulation(W=weighted_random_dag, n=2000, method='linear', sem_type='gauss')
#true_dag, X = dataset.B, dataset.X

if False:
    A = names_graphs_train['00016']
    print(X_train)
    X = X_train['00016']
    print("y_true:", A)
    print("X:", X.head())

    ga = GAE(input_dim=10)
    ga.learn(X)

    # plot est_dag and true_dag
    GraphDAG(ga.causal_matrix, true_dag)

    # calculate accuracy
    met = MetricsDAG(ga.causal_matrix, true_dag)
    print(met.metrics)

In [41]:
if False:
    #shape 16
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=11,
        n_jobs=-1,
        class_weight="balanced"
    )
    display(model)

    print("X_train shape:", X_train_feat.shape)
    print("X_train data types:", X_train_feat.dtypes)
    print("y_train shape:", y_train_feat.shape)
    print("y_train data type:", y_train_feat.dtype)
    print("group_train shape:", group_train.shape)
    print("group_train data type:", group_train.dtype)

    cv = GroupKFold(n_splits=4)
    results = cross_val_score(
        model,
        X_train_feat,
        y_train_feat,
        groups=group_train,
        cv=cv,
        verbose=True,
        scoring="balanced_accuracy"
    )

    print(f"multiclass balanced accuracy: mean={results.mean()}")

In [42]:
if False:

    # Clean feature names
    X_train_feat.columns = [col.replace(' ', '_').replace('"', '').replace("'", '') for col in X_train_feat.columns]
    X_train_feat = X_train_feat.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))

    # Check for constant features
    constant_features = [col for col in X_train_feat.columns if X_train_feat[col].nunique() == 1]
    if constant_features:
        print(f"Constant features: {constant_features}")
        X_train_feat = X_train_feat.drop(columns=constant_features)

    # Check for missing values
    missing_values = X_train_feat.isnull().sum()
    if missing_values.any():
        print(f"Missing values: {missing_values[missing_values > 0]}")
        X_train_feat = X_train_feat.fillna(0)  # Fill missing values with 0 or use an appropriate strategy

    # Define the model
    model = lgb.LGBMClassifier(
        n_estimators=200,
        max_depth=11,
        n_jobs=-1,
        class_weight="balanced",
        verbose=0,
        min_child_samples=20,  # Minimum number of data points in a leaf
        min_split_gain=0.01,   # Minimum gain to make a split
        learning_rate=0.1      # Learning rate
    )
    display(model)

    # Print the shapes of the training data
    print("X_train shape:", X_train_feat.shape)
    print("y_train shape:", y_train_feat.shape)
    print("group_train shape:", group_train.shape)

    # Define the cross-validation strategy
    cv = GroupKFold(n_splits=4)

    # Perform cross-validation
    results = cross_val_score(
        model,
        X_train_feat,
        y_train_feat,
        groups=group_train,
        cv=cv,
        verbose=True,
        scoring="balanced_accuracy"
    )

    # Print the mean balanced accuracy
    print(f"Multiclass balanced accuracy: mean={results.mean()}")

In [43]:
if False:
    x16 = X_train['00016']
    y16 = y_train['00016']

    count = 0
    for name, dataset in X_train.items():
        # Check if the dataset has exactly three columns named 'X', 'Y', and '0'
        if set(dataset.columns) == {'X', 'Y', '0'}:
            print(f"First dataset with columns 'X', 'Y', and '0' is: {name}")
            x16 = dataset
            if count == 2:
                break
            count += 1

    sorted_x16 = x16.sort_values(by='0').reset_index(drop=True)

    plt.figure(figsize=(10, 6))
    plt.plot(sorted_x16['0'], sorted_x16['0'], marker='o', label='Y')
    plt.plot(sorted_x16['0'], sorted_x16['Y'], marker='o', label='Y')
    plt.plot(sorted_x16['0'], sorted_x16['X'], marker='o', label='X')
    plt.xlabel('Sorted Variable 0')
    plt.ylabel('Y')
    plt.title('Sorted Variable 0 vs Y')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Print the sorted DataFrame
    print(sorted_x16)

    print(x16.corr())

In [44]:
def create_all_columns_with_feature_map(functions_names_datasets, n_jobs=-1):
    """
    given a dictionary of {function1:names, function2:names,...} apply
    the desired functions to the list of datasets and merge all of them
    in a single X_y_group dataframe.
    
    Returns:
    - tuple:
        - merged_columns (pd.DataFrame): Merged DataFrame containing all features.
        - feature_map (dict): Dictionary mapping each feature function name to its generated feature columns.
    """

    columns = []
    feature_map = {}
    for function, names_datasets in functions_names_datasets.items():
        print(f"set: {function.__name__}")

        if n_jobs != 1:
            feature_set = create_some_columns_parallel(names_datasets, function, n_jobs=n_jobs)
        else:
            feature_set = create_some_columns(names_datasets, function)
    
    	# Identify feature columns generated by this function, excluding 'dataset' and 'variable'
        generated_feature_cols = [col for col in feature_set.columns if col not in ['dataset', 'variable']]
        feature_map[function.__name__] = generated_feature_cols

        columns.append(feature_set)

    # Merge all feature sets into a single dataframe:
    columns = functools.reduce(
        lambda left, right: pd.merge(left, right, on=["dataset", "variable"]),
        columns,
    )
    
    return columns, feature_map

In [46]:
names_datasets_train = X_train
names_graphs_train = y_train

if False:

    feature_functions = {
        linear_regression_feature: names_datasets_train,
        conditional_distribution_similarity_kl: names_datasets_train,
        conditional_distribution_similarity_jensen_shannon: names_datasets_train,
        conditional_distribution_similarity_hellinger: names_datasets_train,
        conditional_distribution_similarity_bhattacharyya: names_datasets_train,
        conditional_distribution_similarity_emd: names_datasets_train,
        energy_distance_features: names_datasets_train,
        pearson_correlation: names_datasets_train,
        spearman_correlation: names_datasets_train,
        kendall_correlation: names_datasets_train,
        pearson_correlation_with_rolling_multi_labeled: names_datasets_train,
        partial_correlation: names_datasets_train,
        partial_correlation_rank: names_datasets_train,
        mutual_information_and_conditional_MI: names_datasets_train,
        compute_copula_features: names_datasets_train,
        compute_information_metrics: names_datasets_train,
        normalized_prediction_error: names_datasets_train,
        igci_metrics: names_datasets_train,
        moment_features: names_datasets_train,
        fit_metrics: names_datasets_train,
        label: names_graphs_train
    }

    # Create the full dataset with all features and get feature_mapping
    print(f"Creating X_y_group_train from {len(names_datasets_train)} datasets and graphs")
    X_y_group_train, feature_mapping = create_all_columns_with_feature_map(feature_functions)

    # Ensure feature names are unique and sanitize them
    X_y_group_train.columns = pd.Index([
        f"{col}_{i}" if X_y_group_train.columns.duplicated()[i] else col 
        for i, col in enumerate(X_y_group_train.columns)
    ])
    X_y_group_train.columns = X_y_group_train.columns.str.replace('[^A-Za-z0-9_]+', '', regex=True)

    for func_name, features in feature_mapping.items():
        sanitized_features = [re.sub(r'[^A-Za-z0-9_]+', '', col) for col in features]
        feature_mapping[func_name] = sanitized_features

    # Adding numeric labels y
    print("Adding numeric labels y")
    le = LabelEncoder()
    X_y_group_train["y"] = le.fit_transform(X_y_group_train["label"])

    # Reordering columns
    X_y_group_train = X_y_group_train[
        ["dataset", "variable"] +
        X_y_group_train.columns.drop(["dataset", "variable", "label", "y"]).tolist() +
        ["label", "y"]
    ]

    # Extracting features and labels
    print("Extracting X_train, y_train, and group")
    X_train_feat = X_y_group_train.drop(["variable", "dataset", "label", "y"], axis="columns")
    y_train_feat = X_y_group_train["y"]
    group_train = X_y_group_train["dataset"]

    print("X_train shape:", X_train_feat.shape)
    print("X_train data types:", X_train_feat.dtypes.unique())
    print("y_train shape:", y_train_feat.shape)
    print("y_train data type:", y_train_feat.dtype)
    print("group_train shape:", group_train.shape)
    print("group_train data type:", group_train.dtype)


In [ ]:
create_feature_file = False
if create_feature_file:
    with open('X_y_group_train.pkl', 'wb') as f:
        pickle.dump(X_y_group_train, f)

    with open('X_feature_mapping.pkl', 'wb') as f:
        pickle.dump(feature_mapping, f)

In [47]:
def update_feature_functions(
    new_feature_functions,
    x_y_group_train_path="X_y_group_train.pkl",
    feature_mapping_path="X_feature_mapping.pkl",
    output_x_y_group_train_path="X_y_group_train_updated.pkl",
    output_feature_mapping_path="feature_mapping_updated.pkl"
):
    """
    Reads the existing X_y_group_train and feature_mapping from disk,
    applies new feature functions by calling create_all_columns_with_feature_map,
    then saves the updated objects back to disk.

    Parameters:
    -----------
    new_feature_functions : dict
        Dictionary of additional feature-generation functions to add.
    returns: tuple: updated feature_mapping, updated X_y_group_train
    """

    # 1) Load existing data and mapping
    with open(x_y_group_train_path, "rb") as f:
        X_y_group_train = pickle.load(f)

    with open(feature_mapping_path, "rb") as f:
        feature_mapping = pickle.load(f)

    # 2) Apply the new feature functions to the existing data    
    new_features_df, new_feature_map = create_all_columns_with_feature_map(
        new_feature_functions
    )

    # 3) Merge the newly generated features into the existing DataFrame
    X_y_group_train_updated = pd.merge(
        X_y_group_train,
        new_features_df,
        on=["dataset", "variable"],
        how="left"
    )

    # 4) Update the existing feature_map by incorporating the new_map entries
    feature_mapping_updated = dict(feature_mapping)  # shallow copy
    for func_name, col_list in new_feature_map.items():
        if func_name in feature_mapping_updated.keys():
            # Merge/extend unique columns
            original_cols = set(feature_mapping_updated[func_name])
            new_cols = set(col_list)
            feature_mapping_updated[func_name] = list(original_cols | new_cols)
        else:
            feature_mapping_updated[func_name] = col_list

    # 5) Save the updated objects
    with open(output_x_y_group_train_path, "wb") as f:
        pickle.dump(X_y_group_train_updated, f)

    with open(output_feature_mapping_path, "wb") as f:
        pickle.dump(feature_mapping_updated, f)

    print(f"Updated X_y_group_train saved to {output_x_y_group_train_path}")
    print(f"Updated feature_mapping saved to {output_feature_mapping_path}")

    return feature_mapping_updated, X_y_group_train_updated

In [51]:
to_exclude = ["linear_regression_feature"]
for func_name in to_exclude:
    excluded_features = feature_mapping.get(func_name, [])
    X_y_group_train_reduced = X_y_group_train.drop(excluded_features, axis='columns')

extra_feat = {linear_regression_feature: names_datasets_train}
feature_mapping, X_y_group_train_reduced = update_feature_functions(extra_feat)

scoring = ['balanced_accuracy', 'roc_auc_ovr']

X_reduced_feat = X_y_group_train_reduced.drop(["variable", "dataset", "label", "y"], axis="columns")
y_reduced_feat = X_y_group_train_reduced["y"]
group_reduced = X_y_group_train_reduced["dataset"]

# Define the cross-validation strategy
cv = GroupKFold(n_splits=4)

# Initialize the model (ensure it's a fresh instance to avoid data leakage)
model_reduced = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=11,
    n_jobs=-1,
    class_weight="balanced",
    verbose=0,
    min_child_samples=20,  # Minimum number of data points in a leaf
    min_split_gain=0.01,   # Minimum gain to make a split
    learning_rate=0.1,     # Learning rate
    verbosity=-1,
    device='cpu')

print(f"Performing cross-validation without features")

# Perform cross-validation with multiple scoring metrics
cv_results = cross_validate(
    model_reduced,
    X_reduced_feat,
    y_reduced_feat,
    groups=group_reduced,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    verbose=1)

print(f"Performance without Functions - Balanced Accuracy: {cv_results['test_balanced_accuracy'].mean()}")

set: linear_regression_feature


100%|██████████| 23500/23500 [03:57<00:00, 99.03it/s] 


Updated X_y_group_train saved to X_y_group_train_updated.pkl
Updated feature_mapping saved to feature_mapping_updated.pkl
Performing cross-validation without features
Performance without Functions - Balanced Accuracy: 0.5031873372291014


In [43]:
model = lgb.LGBMClassifier(
        n_estimators=200,
        max_depth=11,
        n_jobs=-1,
        class_weight="balanced",
        verbose=0,
        min_child_samples=20,  # Minimum number of data points in a leaf
        min_split_gain=0.01,   # Minimum gain to make a split
        learning_rate=0.1,     # Learning rate
        verbosity=-1,
        device='cpu')

cv = GroupKFold(n_splits=4)
results = []

scoring = ['balanced_accuracy', 'roc_auc_ovr']
results = cross_validate(
    model,
    X_train_feat,
    y_train_feat,
    groups=group_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    verbose=1
)

mean_balanced_accuracy  = results['test_balanced_accuracy'].mean()
print(f"Mean Balanced Accuracy: {mean_balanced_accuracy}")
print(f"Mean ROC AUC: {results['test_roc_auc_ovr'].mean()}")

feature_importance = {} # Dictionary to store performance results

scoring = {
    'Balanced_Accuracy': 'balanced_accuracy',
    'ROC_AUC_OVR': 'roc_auc_ovr'
}

# Perform leave-one-out feature importance
for func, datasets in feature_functions.items():
    func_name = func.__name__
    if func_name == 'label':
        print(f"\nSkipping feature importance evaluation for non-feature function: {func_name}")
        continue  # Skip label as it's not a feature function

    print(f"\nEvaluating feature importance without: {func_name}")

    # Identify the features to exclude based on feature_mapping
    excluded_features = feature_mapping.get(func_name, [])

    if not excluded_features:
        print(f"No features found to exclude for function: {func_name}")
        continue

    print(f"Excluded Features: {excluded_features}")

    # Create dataset excluding features from the current function
    X_y_group_train_reduced = X_y_group_train.drop(excluded_features, axis='columns')

    # Extract features and labels
    X_reduced_feat = X_y_group_train_reduced.drop(["variable", "dataset", "label", "y"], axis="columns")
    y_reduced_feat = X_y_group_train_reduced["y"]
    group_reduced = X_y_group_train_reduced["dataset"]

    # Define the cross-validation strategy
    cv = GroupKFold(n_splits=4)

    # Initialize the model (ensure it's a fresh instance to avoid data leakage)
    model_reduced = lgb.LGBMClassifier(
        n_estimators=200,
        max_depth=11,
        n_jobs=-1,
        class_weight="balanced",
        verbose=0,
        min_child_samples=20,  # Minimum number of data points in a leaf
        min_split_gain=0.01,   # Minimum gain to make a split
        learning_rate=0.1,     # Learning rate
        verbosity=-1,
        device='cpu')

    print(f"Performing cross-validation without features from: {func_name}")

    # Perform cross-validation with multiple scoring metrics
    cv_results = cross_validate(
        model_reduced,
        X_reduced_feat,
        y_reduced_feat,
        groups=group_reduced,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        verbose=1
    )

    # Calculate mean scores
    mean_balanced_accuracy_reduced = mean_balanced_accuracy - cv_results['test_Balanced_Accuracy'].mean()
    mean_roc_auc_reduced = cv_results['test_ROC_AUC_OVR'].mean()

    print(f"Performance without {func_name} - Balanced Accuracy: {mean_balanced_accuracy_reduced}, ROC AUC (OVR): {mean_roc_auc_reduced}")

    # Compute the performance drop compared to the baseline
    performance_drop_balanced_accuracy = mean_balanced_accuracy - mean_balanced_accuracy_reduced
    performance_drop_roc_auc = mean_roc_auc - mean_roc_auc_reduced

    # Store the results in the feature_importance dictionary
    feature_importance[func_name] = {
        'Performance_Drop_Balanced_Accuracy': performance_drop_balanced_accuracy,
        'Performance_Drop_ROC_AUC_OVR': performance_drop_roc_auc,
        'Balanced_Accuracy_Reduced': mean_balanced_accuracy_reduced,
        'ROC_AUC_OVR_Reduced': mean_roc_auc_reduced,
        'Excluded_Features': excluded_features
    }

# Convert the feature_importance dictionary to a DataFrame with 'Excluded_Feature_Function'
importance_df = pd.DataFrame.from_dict(
    feature_importance, orient='index'
).reset_index().rename(columns={'index': 'Excluded_Feature_Function'})

# Expand the 'Excluded_Features' list into a single string separated by commas
importance_df['Excluded_Features'] = importance_df['Excluded_Features'].apply(
    lambda x: ', '.join(x) if isinstance(x, list) else ''
)

# Rename columns appropriately (ensure these match the dictionary keys)
importance_df = importance_df.rename(columns={
    'Performance_Drop_ROC_AUC_OVR': 'Performance_Drop_ROC_AUC_OVR',
    'Performance_Drop_Balanced_Accuracy': 'Performance_Drop_Balanced_Accuracy',
    'Balanced_Accuracy_Reduced': 'Balanced_Accuracy_Reduced',
    'ROC_AUC_OVR_Reduced': 'ROC_AUC_OVR_Reduced',
    'Excluded_Features': 'Excluded_Features'  # No change needed
})

# Sort the DataFrame by 'Performance Drop ROC_AUC' in descending order
#importance_df = importance_df.sort_values(by='Performance_Drop_ROC_AUC', ascending=False).reset_index(drop=True)
importance_df = importance_df.sort_values(by='Balanced_Accuracy_Reduced', ascending=False).reset_index(drop=True)

print("\nFeature Importance based on Performance Drop:")
display(importance_df)

Mean Balanced Accuracy: 0.495165706210293
Mean ROC AUC: 0.8906591543229132

Evaluating feature importance without: linear_regression_feature
Excluded Features: ['fit1_r2', 'fit1_mse', 'fit2_r2', 'fit2_mse', 'fit3_r2', 'fit3_mse']
Performing cross-validation without features from: linear_regression_feature
Performance without linear_regression_feature - Balanced Accuracy: 0.007659259149404296, ROC AUC (OVR): 0.8858945516394884


NameError: name 'mean_roc_auc' is not defined

In [ ]:
to_exclude = ["normalized_prediction_error", "conditional_distribution_similarity_jensen_shannon", "compute_information_metrics", "pearson_correlation_with_rolling_multi_labeled", "energy_distance_features"]
for func_name in to_exclude:
    excluded_features = feature_mapping.get(func_name, [])
    X_y_group_train_reduced = X_y_group_train.drop(excluded_features, axis='columns')

scoring = ['balanced_accuracy', 'roc_auc_ovr']

X_reduced_feat = X_y_group_train_reduced.drop(["variable", "dataset", "label", "y"], axis="columns")
y_reduced_feat = X_y_group_train_reduced["y"]
group_reduced = X_y_group_train_reduced["dataset"]

# Define the cross-validation strategy
cv = GroupKFold(n_splits=4)

# Initialize the model (ensure it's a fresh instance to avoid data leakage)
model_reduced = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=11,
    n_jobs=-1,
    class_weight="balanced",
    verbose=0,
    min_child_samples=20,  # Minimum number of data points in a leaf
    min_split_gain=0.01,   # Minimum gain to make a split
    learning_rate=0.1,     # Learning rate
    verbosity=-1,
    device='cpu')

print(f"Performing cross-validation without features")

# Perform cross-validation with multiple scoring metrics
cv_results = cross_validate(
    model_reduced,
    X_reduced_feat,
    y_reduced_feat,
    groups=group_reduced,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    verbose=1)

print(f"Performance without Functions - Balanced Accuracy: {cv_results['test_balanced_accuracy'].mean()}")

Performing cross-validation without features from: energy_distance_features


KeyError: 'test_Balanced_Accuracy'

In [ ]:
#import tensorflow as tf
#print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
import sys
print(sys.version)
if sys.version_info[0] < 3:
    raise Exception("Must be using Python 3")

3.12.3 (main, Nov  6 2024, 18:32:19) [GCC 13.2.0]
